# CSCI 567 — final: Temporally Robust and Fair Credit Risk Prediction
**LendingClub 2007–2018**


## 0. Setup Environment & Data Load

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!cp "/content/drive/MyDrive/Colab Notebooks/CSCI567/lendingclub/archive.zip" /content/
!unzip -q /content/archive.zip -d /content/

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/content/accepted_2007_to_2018Q4.csv.gz', low_memory=False)
# df = pd.read_csv(
#     '/content/drive/MyDrive/Colab Notebooks/CSCI567/accepted_2007_to_2018Q4.csv',
#     low_memory=False
# )
print(f'Raw shape: {df.shape}')

Raw shape: (2260701, 151)


In [8]:
!pip install optuna -q

In [24]:
import subprocess
try:
    subprocess.check_output('nvidia-smi', shell=True)
    DEVICE = 'cuda'
except:
    DEVICE = 'cpu'

print(f"Device: {DEVICE}")

Device: cuda


In [25]:
import os
print(os.cpu_count())

26


## 1. Preprocessing

In [9]:
# Target Creation
default_labels = ['Charged Off', 'Default',
                  'Does not meet the credit policy. Status:Charged Off']
df['target'] = df['loan_status'].isin(default_labels).astype(int)

# Date Parsing
df['issue_d']  = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['issue_year'] = df['issue_d'].dt.year
df['age_proxy'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')
df['credit_history_years'] = (df['issue_d'] - df['age_proxy']).dt.days / 365

# Feature Selection
FEATURES = ['loan_amnt', 'int_rate', 'installment', 'annual_inc',
            'dti', 'delinq_2yrs', 'fico_range_low', 'open_acc',
            'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
            'mort_acc', 'pub_rec_bankruptcies']

# Sampling
df_sample = df.sample(n=200000, random_state=42).copy()
print(f'Sample shape: {df_sample.shape}')
print(f'Default rate: {df_sample["target"].mean():.3f}')

Sample shape: (200000, 155)
Default rate: 0.119


## 2. Data Splitting

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Temporal split: train / val / test
train_t = df_sample[df_sample['issue_year'] <= 2013].copy()
val_t   = df_sample[(df_sample['issue_year'] >= 2014) & (df_sample['issue_year'] <= 2015)].copy()
test_t  = df_sample[df_sample['issue_year'] >= 2016].copy()

# Imputation: learn medians from training splits only
temporal_fill = train_t[FEATURES + ['credit_history_years']].median()

for split_df in [train_t, val_t, test_t]:
    split_df[FEATURES + ['credit_history_years']] = split_df[FEATURES + ['credit_history_years']].fillna(temporal_fill)

# Fairness groups: learn cutoffs from temporal train only
credit_bins = pd.qcut(train_t['credit_history_years'], q=3, retbins=True, duplicates='drop')[1]
income_clip_top = train_t['annual_inc'].quantile(0.99)
income_bins = pd.qcut(train_t['annual_inc'].clip(upper=income_clip_top), q=3, retbins=True, duplicates='drop')[1]

credit_bins[0] = -np.inf
credit_bins[-1] = np.inf
income_bins[0] = -np.inf
income_bins[-1] = np.inf

for split_df in [train_t, val_t, test_t]:
    split_df['credit_group'] = pd.cut(split_df['credit_history_years'], bins=credit_bins, labels=['short', 'mid', 'long'], include_lowest=True)
    split_df['income_group'] = pd.cut(split_df['annual_inc'].clip(upper=income_clip_top), bins=income_bins, labels=['low', 'mid', 'high'], include_lowest=True)

X_train_t = train_t[FEATURES]
y_train_t = train_t['target']
X_val_t   = val_t[FEATURES]
y_val_t   = val_t['target']
X_test_t  = test_t[FEATURES]
y_test_t  = test_t['target']

# Scaling: fit each scaler on its own training split only
scaler_t = StandardScaler()
X_train_t_sc = scaler_t.fit_transform(X_train_t)
X_val_t_sc   = scaler_t.transform(X_val_t)
X_test_t_sc  = scaler_t.transform(X_test_t)

print('Temporal split')
print(f'  Train (<=2013):   {len(X_train_t):,}')
print(f'  Val (2014-2015):  {len(X_val_t):,}')
print(f'  Test (>=2016):    {len(X_test_t):,}')

print('\nTemporal-train fairness group distribution')
print(train_t['credit_group'].value_counts())
print(train_t['income_group'].value_counts())

print('\nYearly default count')
for yr in sorted(df_sample['issue_year'].unique()):
    sub = df_sample[df_sample['issue_year'] == yr]
    print(f'  {yr}: total={len(sub):,}  default={sub["target"].sum():,}  default_rate={sub["target"].mean():.3f}')

Temporal split
  Train (<=2013):   20,286
  Val (2014-2015):  58,114
  Test (>=2016):    121,596

Temporal-train fairness group distribution
credit_group
mid      6775
short    6764
long     6747
Name: count, dtype: int64
income_group
low     7220
high    6728
mid     6338
Name: count, dtype: int64

Yearly default count
  2007.0: total=46  default=10  default_rate=0.217
  2008.0: total=206  default=37  default_rate=0.180
  2009.0: total=488  default=66  default_rate=0.135
  2010.0: total=1,121  default=176  default_rate=0.157
  2011.0: total=1,946  default=284  default_rate=0.146
  2012.0: total=4,597  default=725  default_rate=0.158
  2013.0: total=11,882  default=1,838  default_rate=0.155
  2014.0: total=20,956  default=3,676  default_rate=0.175
  2015.0: total=37,158  default=6,797  default_rate=0.183
  2016.0: total=38,347  default=5,976  default_rate=0.156
  2017.0: total=39,269  default=3,521  default_rate=0.090
  2018.0: total=43,980  default=746  default_rate=0.017
  nan: total

In [11]:
# By year temporal test sets (AUC over time)
yearly_tests = {}
for yr in sorted(df_sample['issue_year'].unique()):
    sub = df_sample[df_sample['issue_year'] == yr].copy()
    if len(sub) > 0:
        sub[FEATURES + ['credit_history_years']] = sub[FEATURES + ['credit_history_years']].fillna(temporal_fill)
        sub['credit_group'] = pd.cut(sub['credit_history_years'], bins=credit_bins, labels=['short', 'mid', 'long'], include_lowest=True)
        sub['income_group'] = pd.cut(sub['annual_inc'].clip(upper=income_clip_top), bins=income_bins, labels=['low', 'mid', 'high'], include_lowest=True)
        yearly_tests[yr] = (sub[FEATURES], sub['target'], scaler_t.transform(sub[FEATURES]), sub)
        print(f'{yr}: {len(sub):,} samples, default rate={sub["target"].mean():.3f}')

2007.0: 46 samples, default rate=0.217
2008.0: 206 samples, default rate=0.180
2009.0: 488 samples, default rate=0.135
2010.0: 1,121 samples, default rate=0.157
2011.0: 1,946 samples, default rate=0.146
2012.0: 4,597 samples, default rate=0.158
2013.0: 11,882 samples, default rate=0.155
2014.0: 20,956 samples, default rate=0.175
2015.0: 37,158 samples, default rate=0.183
2016.0: 38,347 samples, default rate=0.156
2017.0: 39,269 samples, default rate=0.090
2018.0: 43,980 samples, default rate=0.017


## 3. Model Definition — Full Model + Hyperparameter Sweep


In [12]:
# Computes a robustness-penalized AUC score by subtracting the AUC fluctuation across years from the mean AUC.
# TRP(Temporally Robust Performance)
LAMBDA = 1.0
VAL_YEARS  = [2014, 2015]
TEST_YEARS = [2016, 2017, 2018]

def compute_auc_score(yearly_aucs, years, lam=LAMBDA):
    aucs = [yearly_aucs[yr] for yr in years if yr in yearly_aucs]
    if len(aucs) == 0:
        return np.nan
    auc_mean = np.mean(aucs)
    auc_drop = max(aucs) - min(aucs)
    return auc_mean - lam * auc_drop

In [13]:
# Computes FPR, TPR, PPR for each demographic group at a given threshold to measure model fairness.
def compute_group_metrics_at_threshold(y_true, y_prob, groups, group_col, threshold=0.5):
    records = []
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    for g in groups:
        mask = (group_col == g).values
        yt = y_true[mask]
        yp = y_pred[mask]
        ypr = y_prob[mask]

        if len(yt) == 0:
            continue

        tp = ((yp == 1) & (yt == 1)).sum()
        fp = ((yp == 1) & (yt == 0)).sum()
        tn = ((yp == 0) & (yt == 0)).sum()
        fn = ((yp == 0) & (yt == 1)).sum()

        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        ppr = yp.mean()

        records.append({
            'group': g,
            'threshold': threshold,
            'FPR': fpr,
            'TPR': tpr,
            'PPR': ppr,
            'n': mask.sum(),
        })
    return pd.DataFrame(records)

In [14]:
# Sweeps across multiple thresholds and computes FPR/TPR/PPR disparity (max-min gap between groups) at each threshold.
def compute_threshold_sweep(y_true, y_prob, groups, group_col, thresholds):
    records = []
    for thr in thresholds:
        df_thr = compute_group_metrics_at_threshold(y_true, y_prob, groups, group_col, threshold=thr)
        if len(df_thr) == 0:
            continue
        records.append({
            'threshold': thr,
            'FPR_disp': df_thr['FPR'].max() - df_thr['FPR'].min(),
            'TPR_disp': df_thr['TPR'].max() - df_thr['TPR'].min(),
            'PPR_disp': df_thr['PPR'].max() - df_thr['PPR'].min(),
        })
    return pd.DataFrame(records)

In [15]:
# Trains and evaluates multiple model configs, computing AUC, F1, accuracy, and yearly AUC robustness scores for each model.
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss
from sklearn.metrics import recall_score, f1_score, precision_score
import time

def evaluate_model_configs(model_configs):
    eval_results = {}
    for name, (model, input_type) in model_configs.items():
        t0 = time.time()
        temporal_model = clone(model)

        if input_type == 'scaled':
            Xtr_t, Xva_t, Xte_t = X_train_t_sc, X_val_t_sc, X_test_t_sc
        else:
            Xtr_t, Xva_t, Xte_t = X_train_t, X_val_t, X_test_t

        temporal_model.fit(Xtr_t, y_train_t)

        y_prob_t_val = temporal_model.predict_proba(Xva_t)[:, 1]
        y_prob_t = temporal_model.predict_proba(Xte_t)[:, 1]

        # val set optimal threshold
        thresholds = np.arange(0.05, 0.50, 0.01)
        best_thr_t = max(thresholds, key=lambda t: f1_score(y_val_t, (y_prob_t_val >= t).astype(int), zero_division=0))

        # apply threshold
        y_pred_t_val = (y_prob_t_val >= best_thr_t).astype(int)
        y_pred_t = (y_prob_t >= best_thr_t).astype(int)

        eval_results[name] = {
            'temporal': {
                'val_y_prob': y_prob_t_val,
                'val_y_pred': y_pred_t_val,
                'val_auc': roc_auc_score(y_val_t, y_prob_t_val),
                'val_acc': accuracy_score(y_val_t, y_pred_t_val),
                'val_brier': brier_score_loss(y_val_t, y_prob_t_val),
                'val_recall': recall_score(y_val_t, y_pred_t_val, zero_division=0),
                'val_precision': precision_score(y_val_t, y_pred_t_val, zero_division=0),
                'val_f1': f1_score(y_val_t, y_pred_t_val, zero_division=0),
                'y_pred': y_pred_t,
                'y_prob': y_prob_t,
                'auc': roc_auc_score(y_test_t, y_prob_t),
                'acc': accuracy_score(y_test_t, y_pred_t),
                'brier': brier_score_loss(y_test_t, y_prob_t),
                'recall': recall_score(y_test_t, y_pred_t, zero_division=0),
                'precision': precision_score(y_test_t, y_pred_t, zero_division=0),
                'f1': f1_score(y_test_t, y_pred_t, zero_division=0),
                'threshold': best_thr_t,
                'model': temporal_model,
            },
            'yearly': {}
        }

        for yr, (X_yr, y_yr, X_yr_sc, _) in yearly_tests.items():
            X_in = X_yr_sc if input_type == 'scaled' else X_yr
            yp = temporal_model.predict_proba(X_in)[:, 1]
            eval_results[name]['yearly'][yr] = roc_auc_score(y_yr, yp)

        yearly_aucs = eval_results[name]['yearly']
        eval_results[name]['auc_drop'] = compute_auc_score(yearly_aucs, TEST_YEARS)
        eval_results[name]['val_auc_drop'] = compute_auc_score(yearly_aucs, VAL_YEARS)

    return eval_results

In [16]:
def print_selection_table(eval_results, sort_key='val_auc', reverse=True, title=None, mode='val'):
    if title:
        print(f'\n=== {title} ===')

    if mode == 'val':
        key_fn = {
            'val_auc': lambda x: x[1]['temporal']['val_auc'],
            'val_auc_drop': lambda x: x[1]['val_auc_drop'],
            'val_recall': lambda x: x[1]['temporal']['val_recall'],
            'val_precision': lambda x: x[1]['temporal']['val_precision'],
            'val_f1': lambda x: x[1]['temporal']['val_f1'],
        }.get(sort_key, lambda x: x[1]['temporal']['val_auc'])

        for name, res in sorted(eval_results.items(), key=key_fn, reverse=reverse):
            print(f"{name:20s}  "
                  f"temp_val_AUC={res['temporal']['val_auc']:.4f}  "
                  f"val_AUC_drop={res['val_auc_drop']:.4f}  "
                  f"val_Recall={res['temporal']['val_recall']:.4f}  "
                  f"val_Precision={res['temporal']['val_precision']:.4f}  "
                  f"val_F1={res['temporal']['val_f1']:.4f}  "
                  f"threshold={res['temporal']['threshold']:.2f}")

    elif mode == 'test':
        key_fn = {
            'test_auc': lambda x: x[1]['temporal']['auc'],
            'test_auc_drop': lambda x: x[1]['auc_drop'],
            'recall': lambda x: x[1]['temporal']['recall'],
            'precision': lambda x: x[1]['temporal']['precision'],
            'f1': lambda x: x[1]['temporal']['f1'],
        }.get(sort_key, lambda x: x[1]['temporal']['auc'])

        for name, res in sorted(eval_results.items(), key=key_fn, reverse=reverse):
            print(f"{name:20s}  "
                  f"temp_test_AUC={res['temporal']['auc']:.4f}  "
                  f"test_AUC_drop={res['auc_drop']:.4f}  "
                  f"Recall={res['temporal']['recall']:.4f}  "
                  f"Precision={res['temporal']['precision']:.4f}  "
                  f"F1={res['temporal']['f1']:.4f}  "
                  f"threshold={res['temporal']['threshold']:.2f}")

In [17]:
def print_best_summary(eval_results, prefix=None, title=None):
    filtered = {k: v for k, v in eval_results.items()
                if (k.startswith(prefix) if prefix else True)}

    best_tss = max(filtered.items(), key=lambda x: x[1]['val_auc_drop'])
    best_f1  = max(filtered.items(), key=lambda x: x[1]['temporal']['val_f1'])

    if title:
        print(f'=== {title} ===')

    print(f"\n★ Best by TSS (robust+val): {best_tss[0]}")
    print(f"   val_TSS={best_tss[1]['val_auc_drop']:.4f}  val_AUC={best_tss[1]['temporal']['val_auc']:.4f}  val_F1={best_tss[1]['temporal']['val_f1']:.4f}")

    print(f"\n★ Best by val F1:           {best_f1[0]}")
    print(f"   val_TSS={best_f1[1]['val_auc_drop']:.4f}  val_AUC={best_f1[1]['temporal']['val_auc']:.4f}  val_F1={best_f1[1]['temporal']['val_f1']:.4f}")

In [18]:
def print_optuna_results(study, top_trials, title='Optuna Results'):
    print(f'\n=== {title} ===')
    print(f'Best score: {study.best_value:.4f}')
    print(f'\nBest params:')
    for k, v in study.best_params.items():
        print(f'  {k:20s}: {v}')

    print(f'\n=== Top {len(top_trials)} Trials ===')
    for i, t in enumerate(top_trials):
        print(f'\n[{i+1}] score={t.value:.4f}')
        for k, v in t.params.items():
            print(f'  {k:20s}: {v}')

## 3.1 XGBoost Test Code (Find Best)


In [19]:
import optuna
from optuna.samplers import TPESampler

w_f1 = 0.5
w_auc = 0.5

## 3.1.1 XGBoost Grid Search


In [62]:
# A dictionary defining 100+ XGB model configurations by systematically combining hyperparameters such as learning_rate, max_depth, subsample, gamma, and L1/L2 regularization.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

XGB_MODEL_CONFIGS = {
    # Baseline
    'XGB_base':         (XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, eval_metric='logloss', random_state=42), 'scaled'),

    # learning_rate sweep
    'XGB_lr0.005':       (XGBClassifier(n_estimators=100, learning_rate=0.005, max_depth=4, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_lr0.01':       (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=4, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_lr0.05':       (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_lr0.3':        (XGBClassifier(n_estimators=100, learning_rate=0.3,  max_depth=4, eval_metric='logloss', random_state=42), 'scaled'),

    # max_depth sweep
    'XGB_d3_lr0.01':           (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_d6_lr0.01':           (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=6, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_d8_lr0.01':           (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=8, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_d3_lr0.03':           (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_d6_lr0.03':           (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=6, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_d8_lr0.03':           (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=8, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_d3_lr0.05':           (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_d6_lr0.05':           (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=6, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_d8_lr0.05':           (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=8, eval_metric='logloss', random_state=42), 'scaled'),

    # min_child_weight sweep
    'XGB_mcw3_lr0.01':         (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3,  eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_mcw5_lr0.01':         (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=5,  eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_mcw10_lr0.01':        (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=10, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_mcw3_lr0.03':         (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3,  eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_mcw5_lr0.03':         (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=5,  eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_mcw10_lr0.03':        (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=10, eval_metric='logloss', random_state=42), 'scaled'),

    # subsample sweep
    'XGB_sub0.6_lr0.01':       (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, subsample=0.6, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_sub0.7_lr0.01':       (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_sub0.8_lr0.01':       (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, subsample=0.8, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_sub0.6_lr0.03':       (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, subsample=0.6, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_sub0.7_lr0.03':       (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_sub0.8_lr0.03':       (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, subsample=0.8, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_sub0.6_lr0.05':       (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, subsample=0.6, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_sub0.7_lr0.05':       (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_sub0.8_lr0.05':       (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, subsample=0.8, eval_metric='logloss', random_state=42), 'scaled'),

    # colsample_bytree sweep
    'XGB_col0.6_lr0.01':       (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, subsample=0.7, colsample_bytree=0.6, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.7_lr0.01':       (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, subsample=0.7, colsample_bytree=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.8_lr0.01':       (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, subsample=0.7, colsample_bytree=0.8, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.6_lr0.03':       (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, subsample=0.7, colsample_bytree=0.6, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.7_lr0.03':       (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, subsample=0.7, colsample_bytree=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.8_lr0.03':       (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, subsample=0.7, colsample_bytree=0.8, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.6_lr0.05':       (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, subsample=0.7, colsample_bytree=0.6, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.7_lr0.05':       (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, subsample=0.7, colsample_bytree=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.8_lr0.05':       (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, subsample=0.7, colsample_bytree=0.8, eval_metric='logloss', random_state=42), 'scaled'),

    # colsample_bytree sweep
    'XGB_col0.6_lr0.01_rmsub':       (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, colsample_bytree=0.6, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.7_lr0.01_rmsub':       (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, colsample_bytree=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.8_lr0.01_rmsub':       (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, colsample_bytree=0.8, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.6_lr0.03_rmsub':       (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, colsample_bytree=0.6, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.7_lr0.03_rmsub':       (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, colsample_bytree=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.8_lr0.03_rmsub':       (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, colsample_bytree=0.8, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.6_lr0.05_rmsub':       (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, colsample_bytree=0.6, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.7_lr0.05_rmsub':       (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, colsample_bytree=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.8_lr0.05_rmsub':       (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, colsample_bytree=0.8, eval_metric='logloss', random_state=42), 'scaled'),

    # colsample_bytree sweep
    'XGB_col0.6_lr0.01_rmmcw':       (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, subsample=0.7, colsample_bytree=0.6, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.7_lr0.01_rmmcw':       (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, subsample=0.7, colsample_bytree=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.8_lr0.01_rmmcw':       (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, subsample=0.7, colsample_bytree=0.8, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.6_lr0.03_rmmcw':       (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, subsample=0.7, colsample_bytree=0.6, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.7_lr0.03_rmmcw':       (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, subsample=0.7, colsample_bytree=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.8_lr0.03_rmmcw':       (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, subsample=0.7, colsample_bytree=0.8, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.6_lr0.05_rmmcw':       (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, subsample=0.7, colsample_bytree=0.6, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.7_lr0.05_rmmcw':       (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, subsample=0.7, colsample_bytree=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_col0.8_lr0.05_rmmcw':       (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, subsample=0.7, colsample_bytree=0.8, eval_metric='logloss', random_state=42), 'scaled'),

    # subsample sweep
    'XGB_gamma1_lr0.01':       (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=1, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_gamma3_lr0.01':       (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=3, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_gamma5_lr0.01':       (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=5, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_gamma1_lr0.03':       (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=1, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_gamma3_lr0.03':       (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=3, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_gamma5_lr0.03':       (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=5, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_gamma1_lr0.05':       (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=1, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_gamma3_lr0.05':       (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=3, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_gamma5_lr0.05':       (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=5, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),

    # L1 (reg_alpha) + gamma + lr 조합
    'XGB_L1_1_g1_lr0.01':  (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=1, reg_alpha=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_5_g1_lr0.01':  (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=1, reg_alpha=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_10_g1_lr0.01': (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=1, reg_alpha=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_1_g3_lr0.01':  (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=3, reg_alpha=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_5_g3_lr0.01':  (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=3, reg_alpha=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_10_g3_lr0.01': (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=3, reg_alpha=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_1_g5_lr0.01':  (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=5, reg_alpha=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_5_g5_lr0.01':  (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=5, reg_alpha=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_10_g5_lr0.01': (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=5, reg_alpha=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),

    'XGB_L1_1_g1_lr0.03':  (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=1, reg_alpha=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_5_g1_lr0.03':  (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=1, reg_alpha=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_10_g1_lr0.03': (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=1, reg_alpha=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_1_g3_lr0.03':  (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=3, reg_alpha=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_5_g3_lr0.03':  (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=3, reg_alpha=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_10_g3_lr0.03': (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=3, reg_alpha=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_1_g5_lr0.03':  (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=5, reg_alpha=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_5_g5_lr0.03':  (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=5, reg_alpha=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_10_g5_lr0.03': (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=5, reg_alpha=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),

    'XGB_L1_1_g1_lr0.05':  (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=1, reg_alpha=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_5_g1_lr0.05':  (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=1, reg_alpha=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_10_g1_lr0.05': (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=1, reg_alpha=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_1_g3_lr0.05':  (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=3, reg_alpha=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_5_g3_lr0.05':  (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=3, reg_alpha=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_10_g3_lr0.05': (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=3, reg_alpha=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_1_g5_lr0.05':  (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=5, reg_alpha=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_5_g5_lr0.05':  (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=5, reg_alpha=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L1_10_g5_lr0.05': (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=5, reg_alpha=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),

    # L2 (reg_lambda) + gamma + lr (27)
    'XGB_L2_1_g1_lr0.01':  (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_5_g1_lr0.01':  (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_10_g1_lr0.01': (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_1_g3_lr0.01':  (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_5_g3_lr0.01':  (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_10_g3_lr0.01': (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_1_g5_lr0.01':  (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_5_g5_lr0.01':  (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_10_g5_lr0.01': (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),

    'XGB_L2_1_g1_lr0.03':  (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_5_g1_lr0.03':  (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_10_g1_lr0.03': (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_1_g3_lr0.03':  (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_5_g3_lr0.03':  (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_10_g3_lr0.03': (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_1_g5_lr0.03':  (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_5_g5_lr0.03':  (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_10_g5_lr0.03': (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),

    'XGB_L2_1_g1_lr0.05':  (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_5_g1_lr0.05':  (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_10_g1_lr0.05': (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_1_g3_lr0.05':  (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_5_g3_lr0.05':  (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_10_g3_lr0.05': (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_1_g5_lr0.05':  (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=1.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_5_g5_lr0.05':  (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=5.0,  subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2_10_g5_lr0.05': (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=10.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),

    'XGB_L2(1)_g1_lr0.01_n50':   (XGBClassifier(n_estimators=50,  learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g1_lr0.01_n75':   (XGBClassifier(n_estimators=75,  learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g1_lr0.01_n100':  (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g1_lr0.03_n50':   (XGBClassifier(n_estimators=50,  learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g1_lr0.03_n75':   (XGBClassifier(n_estimators=75,  learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g1_lr0.03_n100':  (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g1_lr0.05_n50':   (XGBClassifier(n_estimators=50,  learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g1_lr0.05_n75':   (XGBClassifier(n_estimators=75,  learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g1_lr0.05_n100':  (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=1, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g3_lr0.01_n50':   (XGBClassifier(n_estimators=50,  learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g3_lr0.01_n75':   (XGBClassifier(n_estimators=75,  learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g3_lr0.01_n100':  (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g3_lr0.03_n50':   (XGBClassifier(n_estimators=50,  learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g3_lr0.03_n75':   (XGBClassifier(n_estimators=75,  learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g3_lr0.03_n100':  (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g3_lr0.05_n50':   (XGBClassifier(n_estimators=50,  learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g3_lr0.05_n75':   (XGBClassifier(n_estimators=75,  learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g3_lr0.05_n100':  (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=3, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g5_lr0.01_n50':   (XGBClassifier(n_estimators=50,  learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g5_lr0.01_n75':   (XGBClassifier(n_estimators=75,  learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g5_lr0.01_n100':  (XGBClassifier(n_estimators=100, learning_rate=0.01, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g5_lr0.03_n50':   (XGBClassifier(n_estimators=50,  learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g5_lr0.03_n75':   (XGBClassifier(n_estimators=75,  learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g5_lr0.03_n100':  (XGBClassifier(n_estimators=100, learning_rate=0.03, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g5_lr0.05_n50':   (XGBClassifier(n_estimators=50,  learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g5_lr0.05_n75':   (XGBClassifier(n_estimators=75,  learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
    'XGB_L2(1)_g5_lr0.05_n100':  (XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, min_child_weight=3, gamma=5, reg_lambda=1.0, subsample=0.7, eval_metric='logloss', random_state=42), 'scaled'),
}

print(f'Total model configs: {len(XGB_MODEL_CONFIGS)}')

Total model configs: 146


In [63]:
xgb_results = evaluate_model_configs(XGB_MODEL_CONFIGS)

# (val set)
print_selection_table(xgb_results, sort_key='val_auc', reverse=True, title='XGB - Val Set', mode='val')
print_selection_table(xgb_results, sort_key='val_auc_drop', reverse=False, title='XGB - Val Robustness', mode='val')

# (test set)
print_selection_table(xgb_results, sort_key='test_auc', reverse=True, title='XGB - Test Set (Final)', mode='test')
print_selection_table(xgb_results, sort_key='test_auc_drop', reverse=False, title='XGB - Test Robustness (Final)', mode='test')

# (best summary)
print_best_summary(xgb_results, prefix='XGB_', title='XGBoost Selection Summary')


=== XGB - Val Set ===
XGB_col0.7_lr0.03     temp_val_AUC=0.6837  val_AUC_drop=0.6675  val_Recall=0.6303  val_Precision=0.2736  val_F1=0.3815  threshold=0.15
XGB_col0.6_lr0.03     temp_val_AUC=0.6837  val_AUC_drop=0.6674  val_Recall=0.6355  val_Precision=0.2720  val_F1=0.3809  threshold=0.15
XGB_col0.6_lr0.03_rmmcw  temp_val_AUC=0.6837  val_AUC_drop=0.6677  val_Recall=0.6359  val_Precision=0.2709  val_F1=0.3800  threshold=0.15
XGB_col0.7_lr0.05_rmmcw  temp_val_AUC=0.6836  val_AUC_drop=0.6680  val_Recall=0.6247  val_Precision=0.2751  val_F1=0.3820  threshold=0.15
XGB_col0.7_lr0.05     temp_val_AUC=0.6836  val_AUC_drop=0.6675  val_Recall=0.5823  val_Precision=0.2832  val_F1=0.3811  threshold=0.16
XGB_col0.7_lr0.03_rmmcw  temp_val_AUC=0.6836  val_AUC_drop=0.6674  val_Recall=0.6291  val_Precision=0.2740  val_F1=0.3818  threshold=0.15
XGB_col0.6_lr0.05     temp_val_AUC=0.6835  val_AUC_drop=0.6685  val_Recall=0.5831  val_Precision=0.2839  val_F1=0.3819  threshold=0.16
XGB_col0.6_lr0.05_rmmcw

In [42]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
import time

param_dist = {
    'learning_rate': [0.01, 0.03, 0.05],
    'max_depth': [3, 4, 5],
    'min_child_weight': [3, 5, 10],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.6, 0.7, 0.8],
    'gamma': [1, 3, 5],
    'reg_alpha': [1.0, 5.0, 10.0],
    'reg_lambda': [1.0, 5.0, 10.0],
    'n_estimators': [50, 75, 100],
}

random_search_xgb = RandomizedSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=42, device=DEVICE),
    param_dist,
    n_iter=2000,
    cv=3,
    scoring='roc_auc',
    n_jobs=22 if DEVICE == 'gpu' else -1,
    verbose=1,
    random_state=42
)

start = time.time()
random_search_xgb.fit(X_train_t_sc, y_train_t)
elapsed = time.time() - start

print(f"\nTotal time: {elapsed/60:.1f} min")
print(f"Best Score: {random_search_xgb.best_score_:.4f}")
print(f"Best Params: {random_search_xgb.best_params_}")

Fitting 3 folds for each of 2000 candidates, totalling 6000 fits

Total time: 16.3 min
Best Score: 0.6736
Best Params: {'subsample': 0.6, 'reg_lambda': 1.0, 'reg_alpha': 1.0, 'n_estimators': 100, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.03, 'gamma': 1, 'colsample_bytree': 0.7}


In [41]:
import pandas as pd

results = pd.DataFrame(random_search_xgb.cv_results_)

params = ['param_learning_rate', 'param_max_depth', 'param_min_child_weight', 'param_subsample',
          'param_colsample_bytree', 'param_gamma', 'param_reg_alpha', 'param_reg_lambda',
          'param_n_estimators']

# params = ['param_learning_rate', 'param_max_depth', 'param_min_child_weight', 'param_reg_alpha', 'param_reg_lambda']
for p in params:
    try:
        grouped = results.groupby(p)['mean_test_score'].mean()
        spread = grouped.max() - grouped.min()
        print(f"{p:30s} | score spread: {spread:.4f}")
        print(grouped.to_string())
        print()
    except:
        print(f"{p:30s} | not found, skipped")
        print()

param_learning_rate            | score spread: 0.0018
param_learning_rate
0.01    0.669147
0.03    0.670416
0.05    0.670930

param_max_depth                | score spread: 0.0000
param_max_depth
3    0.670194
4    0.670209
5    0.670178

param_min_child_weight         | score spread: 0.0001
param_min_child_weight
3     0.670256
5     0.670117
10    0.670216

param_subsample                | score spread: 0.0004
param_subsample
0.6    0.670347
0.7    0.669959
0.8    0.670276

param_colsample_bytree         | score spread: 0.0011
param_colsample_bytree
0.6    0.669757
0.7    0.670058
0.8    0.670840

param_gamma                    | score spread: 0.0002
param_gamma
1    0.670273
3    0.670265
5    0.670048

param_reg_alpha                | score spread: 0.0015
param_reg_alpha
1.0     0.670898
5.0     0.670246
10.0    0.669423

param_reg_lambda               | score spread: 0.0003
param_reg_lambda
1.0     0.670331
5.0     0.670223
10.0    0.670027

param_n_estimators             | score 

## 3.1.2 XGBoost Bayesian Search

In [52]:
# Optuna hyperparameter tuning for XGBoost, optimizing a weighted combination of AUC robustness score and F1 over 60 trials.
def objective_xgb(trial):
    params = {
        # Fixed parameters: Based on the highest average performance in Phase 1 (Randomized Search)
        # 'max_depth': 3,
        # 'min_child_weight': 3,
        # 'gamma': 1,
        # 'reg_lambda': 1.0,
        # 'subsample': trial.suggest_float('subsample', 0.6, 0.85),

        # Fine-tuning parameters: Targeting variables with high performance sensitivity (Score Spread)
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.1, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 3.0), # Optimized range based on search results
        'n_estimators': trial.suggest_int('n_estimators', 75, 200),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.9),
    }

    model = XGBClassifier(**params, eval_metric='logloss', random_state=42, device=DEVICE)
    model.fit(X_train_t_sc, y_train_t)

    yearly_aucs = {}
    for yr, (X_yr, y_yr, X_yr_sc, _) in yearly_tests.items():
        yp = model.predict_proba(X_yr_sc)[:, 1]
        yearly_aucs[yr] = roc_auc_score(y_yr, yp)

    auc_drop = compute_auc_score(yearly_aucs, VAL_YEARS)

    y_prob_val = model.predict_proba(X_val_t_sc)[:, 1]
    thresholds = np.arange(0.05, 0.50, 0.01)
    best_thr = max(thresholds, key=lambda t: f1_score(y_val_t, (y_prob_val >= t).astype(int), zero_division=0))
    val_f1 = f1_score(y_val_t, (y_prob_val >= best_thr).astype(int), zero_division=0)

    return w_auc * auc_drop + w_f1 * val_f1

study_xgb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=180, show_progress_bar=True)

[I 2026-05-09 09:51:23,359] A new study created in memory with name: no-name-c926232e-eb9b-421b-8e61-1c0df73f93a3


  0%|          | 0/180 [00:00<?, ?it/s]

[I 2026-05-09 09:51:24,134] Trial 0 finished with value: 0.51589948419724 and parameters: {'learning_rate': 0.04709342990753529, 'reg_alpha': 2.8521429192297485, 'n_estimators': 167, 'colsample_bytree': 0.8197316968394073}. Best is trial 0 with value: 0.51589948419724.
[I 2026-05-09 09:51:24,642] Trial 1 finished with value: 0.5171765212029292 and parameters: {'learning_rate': 0.036199292763564, 'reg_alpha': 0.46798356100860794, 'n_estimators': 82, 'colsample_bytree': 0.8732352291549871}. Best is trial 1 with value: 0.5171765212029292.
[I 2026-05-09 09:51:25,046] Trial 2 finished with value: 0.5174020765411179 and parameters: {'learning_rate': 0.06186307704341194, 'reg_alpha': 2.1242177333881367, 'n_estimators': 77, 'colsample_bytree': 0.8939819704323989}. Best is trial 2 with value: 0.5174020765411179.
[I 2026-05-09 09:51:25,405] Trial 3 finished with value: 0.5124431952276349 and parameters: {'learning_rate': 0.08173118924709635, 'reg_alpha': 0.6370173320348285, 'n_estimators': 97, '

In [53]:
# Extracts best Optuna trials
# best
# Random search
trials_df = study_xgb.trials_dataframe().sort_values('value', ascending=False).head(5)

XGB_MODEL_CONFIGS = {}
for i, row in enumerate(study_xgb.trials):
    if i >= 10:
        break
top_trials = sorted(study_xgb.trials, key=lambda t: t.value if t.value else -999, reverse=True)[:10]

for i, t in enumerate(top_trials):
    name = f'XGB_optuna_{i+1}'
    XGB_MODEL_CONFIGS[name] = (
        XGBClassifier(**t.params, eval_metric='logloss', random_state=42), 'raw'
    )

# evaluation
xgb_results = evaluate_model_configs(XGB_MODEL_CONFIGS)

# val set
print_selection_table(xgb_results, sort_key='val_auc', reverse=True, title='XGB - Val Set', mode='val')
print_selection_table(xgb_results, sort_key='val_auc_drop', reverse=False, title='TRP', mode='val')
print_selection_table(xgb_results, sort_key='val_f1', reverse=True, title='XGB - Val F1', mode='val')

# best summary
print_best_summary(xgb_results, prefix='XGB_', title='XGBoost Selection Summary')

# Hyperparameter print
print_optuna_results(study_xgb, top_trials, title='XGB Optuna Hyperparameter Results')


=== XGB - Val Set ===
XGB_optuna_4          temp_val_AUC=0.6784  val_AUC_drop=0.6627  val_Recall=0.5953  val_Precision=0.2738  val_F1=0.3751  threshold=0.16
XGB_optuna_10         temp_val_AUC=0.6782  val_AUC_drop=0.6628  val_Recall=0.6480  val_Precision=0.2652  val_F1=0.3764  threshold=0.15
XGB_optuna_6          temp_val_AUC=0.6781  val_AUC_drop=0.6631  val_Recall=0.6015  val_Precision=0.2750  val_F1=0.3774  threshold=0.16
XGB_optuna_8          temp_val_AUC=0.6780  val_AUC_drop=0.6618  val_Recall=0.6420  val_Precision=0.2646  val_F1=0.3748  threshold=0.15
XGB_optuna_3          temp_val_AUC=0.6778  val_AUC_drop=0.6627  val_Recall=0.6435  val_Precision=0.2648  val_F1=0.3752  threshold=0.15
XGB_optuna_7          temp_val_AUC=0.6777  val_AUC_drop=0.6619  val_Recall=0.5980  val_Precision=0.2743  val_F1=0.3761  threshold=0.16
XGB_optuna_1          temp_val_AUC=0.6774  val_AUC_drop=0.6616  val_Recall=0.6426  val_Precision=0.2652  val_F1=0.3755  threshold=0.15
XGB_optuna_2          temp_val_A

In [65]:
# Optuna hyperparameter tuning for XGBoost, optimizing a weighted combination of AUC robustness score and F1 over 60 trials.
# Manual search
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 4),
        'subsample': trial.suggest_float('subsample', 0.6, 0.85),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 3.0),
    }

    model = XGBClassifier(**params, eval_metric='logloss', random_state=42, device=DEVICE)
    model.fit(X_train_t_sc, y_train_t)

    yearly_aucs = {}
    for yr, (X_yr, y_yr, X_yr_sc, _) in yearly_tests.items():
        yp = model.predict_proba(X_yr_sc)[:, 1]
        yearly_aucs[yr] = roc_auc_score(y_yr, yp)

    auc_drop = compute_auc_score(yearly_aucs, VAL_YEARS)

    y_prob_val = model.predict_proba(X_val_t_sc)[:, 1]
    thresholds = np.arange(0.05, 0.50, 0.01)
    best_thr = max(thresholds, key=lambda t: f1_score(y_val_t, (y_prob_val >= t).astype(int), zero_division=0))
    val_f1 = f1_score(y_val_t, (y_prob_val >= best_thr).astype(int), zero_division=0)

    return w_auc * auc_drop + w_f1 * val_f1

study_xgb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=120, show_progress_bar=True)

[I 2026-05-09 10:02:11,833] A new study created in memory with name: no-name-ed252657-ee1f-4887-aa84-36cdc666e4c6


  0%|          | 0/120 [00:00<?, ?it/s]

[I 2026-05-09 10:02:12,371] Trial 0 finished with value: 0.5234953540891498 and parameters: {'n_estimators': 87, 'learning_rate': 0.046187109390049115, 'max_depth': 4, 'subsample': 0.7496646210492591, 'gamma': 0.7800932022121826, 'reg_lambda': 1.3119890406724053}. Best is trial 0 with value: 0.5234953540891498.
[I 2026-05-09 10:02:12,702] Trial 1 finished with value: 0.5219406719303155 and parameters: {'n_estimators': 55, 'learning_rate': 0.04031170288036924, 'max_depth': 4, 'subsample': 0.7770181444490114, 'gamma': 0.10292247147901223, 'reg_lambda': 2.9398197043239884}. Best is trial 0 with value: 0.5234953540891498.
[I 2026-05-09 10:02:13,027] Trial 2 finished with value: 0.52303750833565 and parameters: {'n_estimators': 134, 'learning_rate': 0.014074036373847479, 'max_depth': 3, 'subsample': 0.6458511274633585, 'gamma': 1.5212112147976886, 'reg_lambda': 2.049512863264476}. Best is trial 0 with value: 0.5234953540891498.
[I 2026-05-09 10:02:13,331] Trial 3 finished with value: 0.5226

In [66]:
# Extracts best Optuna trials
# best manual search
trials_df = study_xgb.trials_dataframe().sort_values('value', ascending=False).head(5)

XGB_MODEL_CONFIGS = {}
for i, row in enumerate(study_xgb.trials):
    if i >= 10:
        break
top_trials = sorted(study_xgb.trials, key=lambda t: t.value if t.value else -999, reverse=True)[:10]

for i, t in enumerate(top_trials):
    name = f'XGB_optuna_{i+1}'
    XGB_MODEL_CONFIGS[name] = (
        XGBClassifier(**t.params, eval_metric='logloss', random_state=42), 'raw'
    )

# evaluation
xgb_results = evaluate_model_configs(XGB_MODEL_CONFIGS)

# val set
print_selection_table(xgb_results, sort_key='val_auc', reverse=True, title='XGB - Val Set', mode='val')
print_selection_table(xgb_results, sort_key='val_auc_drop', reverse=False, title='TRP', mode='val')
print_selection_table(xgb_results, sort_key='val_f1', reverse=True, title='XGB - Val F1', mode='val')

# best summary
print_best_summary(xgb_results, prefix='XGB_', title='XGBoost Selection Summary')

# Hyperparameter print
print_optuna_results(study_xgb, top_trials, title='XGB Optuna Hyperparameter Results')


=== XGB - Val Set ===
XGB_optuna_4          temp_val_AUC=0.6835  val_AUC_drop=0.6694  val_Recall=0.5895  val_Precision=0.2831  val_F1=0.3826  threshold=0.16
XGB_optuna_10         temp_val_AUC=0.6835  val_AUC_drop=0.6685  val_Recall=0.5876  val_Precision=0.2833  val_F1=0.3823  threshold=0.16
XGB_optuna_7          temp_val_AUC=0.6833  val_AUC_drop=0.6687  val_Recall=0.5493  val_Precision=0.2922  val_F1=0.3815  threshold=0.17
XGB_optuna_9          temp_val_AUC=0.6833  val_AUC_drop=0.6679  val_Recall=0.5853  val_Precision=0.2830  val_F1=0.3815  threshold=0.16
XGB_optuna_8          temp_val_AUC=0.6827  val_AUC_drop=0.6683  val_Recall=0.5888  val_Precision=0.2812  val_F1=0.3807  threshold=0.16
XGB_optuna_6          temp_val_AUC=0.6825  val_AUC_drop=0.6689  val_Recall=0.5975  val_Precision=0.2799  val_F1=0.3813  threshold=0.16
XGB_optuna_5          temp_val_AUC=0.6825  val_AUC_drop=0.6684  val_Recall=0.6241  val_Precision=0.2732  val_F1=0.3800  threshold=0.15
XGB_optuna_3          temp_val_A

## 3.2 Logistic Regression Test Code (Find Best)


## 3.2.1 LR Gird Search

In [67]:
# A dictionary defining LR model configurations by sweeping L1, L2, and ElasticNet regularization with various C and l1_ratio values.
LR_MODEL_CONFIGS = {
    # L2 sweep
    'LR_L2_C0.0001': (LogisticRegression(C=0.0001, penalty='l2', solver='liblinear', max_iter=2000, random_state=42), 'scaled'),
    'LR_L2_C0.001':  (LogisticRegression(C=0.001,  penalty='l2', solver='liblinear', max_iter=2000, random_state=42), 'scaled'),
    'LR_L2_C0.01':   (LogisticRegression(C=0.01,   penalty='l2', solver='liblinear', max_iter=2000, random_state=42), 'scaled'),
    'LR_L2_C0.1':    (LogisticRegression(C=0.1,    penalty='l2', solver='liblinear', max_iter=2000, random_state=42), 'scaled'),
    'LR_L2_C1':      (LogisticRegression(C=1.0,    penalty='l2', solver='liblinear', max_iter=2000, random_state=42), 'scaled'),
    'LR_L2_C10':     (LogisticRegression(C=10.0,   penalty='l2', solver='liblinear', max_iter=2000, random_state=42), 'scaled'),
    'LR_L2_C100':    (LogisticRegression(C=100.0,  penalty='l2', solver='liblinear', max_iter=2000, random_state=42), 'scaled'),

    # L1 sweep (sparsity)
    'LR_L1_C0.001':  (LogisticRegression(C=0.0001,  penalty='l1', solver='liblinear', max_iter=2000, random_state=42), 'scaled'),
    'LR_L1_C0.01':   (LogisticRegression(C=0.001,   penalty='l1', solver='liblinear', max_iter=2000, random_state=42), 'scaled'),
    'LR_L1_C0.1':    (LogisticRegression(C=0.01,    penalty='l1', solver='liblinear', max_iter=2000, random_state=42), 'scaled'),
    'LR_L1_C1':      (LogisticRegression(C=1.0,    penalty='l1', solver='liblinear', max_iter=2000, random_state=42), 'scaled'),
    'LR_L1_C10':     (LogisticRegression(C=10.0,   penalty='l1', solver='liblinear', max_iter=2000, random_state=42), 'scaled'),
    'LR_L1_C100':    (LogisticRegression(C=100.0,  penalty='l1', solver='liblinear', max_iter=2000, random_state=42), 'scaled'),

    # ElasticNet sweep (L1+L2)
    'LR_EN_C0.0001_l10.3':  (LogisticRegression(C=0.0001,  penalty='elasticnet', solver='saga', l1_ratio=0.3, max_iter=2000, random_state=42), 'scaled'),
    'LR_EN_C0.0001_l10.5':  (LogisticRegression(C=0.0001,  penalty='elasticnet', solver='saga', l1_ratio=0.5, max_iter=2000, random_state=42), 'scaled'),
    'LR_EN_C0.0001_l10.7':  (LogisticRegression(C=0.0001,  penalty='elasticnet', solver='saga', l1_ratio=0.7, max_iter=2000, random_state=42), 'scaled'),
    'LR_EN_C0.001_l10.05':   (LogisticRegression(C=0.001,   penalty='elasticnet', solver='saga', l1_ratio=0.05, max_iter=2000, random_state=42), 'scaled'),
    'LR_EN_C0.001_l10.1':   (LogisticRegression(C=0.001,   penalty='elasticnet', solver='saga', l1_ratio=0.1, max_iter=2000, random_state=42), 'scaled'),
    'LR_EN_C0.001_l10.2':   (LogisticRegression(C=0.001,   penalty='elasticnet', solver='saga', l1_ratio=0.2, max_iter=2000, random_state=42), 'scaled'),
    'LR_EN_C0.001_l10.3':   (LogisticRegression(C=0.001,   penalty='elasticnet', solver='saga', l1_ratio=0.3, max_iter=2000, random_state=42), 'scaled'),
    'LR_EN_C0.001_l10.5':   (LogisticRegression(C=0.001,   penalty='elasticnet', solver='saga', l1_ratio=0.5, max_iter=2000, random_state=42), 'scaled'),
    'LR_EN_C0.001_l10.7':   (LogisticRegression(C=0.001,   penalty='elasticnet', solver='saga', l1_ratio=0.7, max_iter=2000, random_state=42), 'scaled'),
    'LR_EN_C0.01_l10.3':     (LogisticRegression(C=1.01,   penalty='elasticnet', solver='saga', l1_ratio=0.3, max_iter=2000, random_state=42), 'scaled'),
    'LR_EN_C0.01_l10.5':     (LogisticRegression(C=1.01,   penalty='elasticnet', solver='saga', l1_ratio=0.5, max_iter=2000, random_state=42), 'scaled'),
    'LR_EN_C0.01_l10.7':     (LogisticRegression(C=1.01,   penalty='elasticnet', solver='saga', l1_ratio=0.7, max_iter=2000, random_state=42), 'scaled'),
}

print(f'Total LR model configs: {len(LR_MODEL_CONFIGS)}')

Total LR model configs: 25


In [68]:
lr_results = evaluate_model_configs(LR_MODEL_CONFIGS)

# (val set)
print_selection_table(lr_results, sort_key='val_auc', reverse=True, title='LR - Val Set', mode='val')
print_selection_table(lr_results, sort_key='val_auc_drop', reverse=False, title='LR - Val Robustness', mode='val')

# (test set)
print_selection_table(lr_results, sort_key='test_auc', reverse=True, title='LR - Test Set (Final)', mode='test')
print_selection_table(lr_results, sort_key='test_auc_drop', reverse=False, title='LR - Test Robustness (Final)', mode='test')

# (best summary)
print_best_summary(lr_results, prefix='LR_', title='Logistic Regression Selection Summary')


=== LR - Val Set ===
LR_L2_C0.001          temp_val_AUC=0.6858  val_AUC_drop=0.6652  val_Recall=0.5870  val_Precision=0.2842  val_F1=0.3830  threshold=0.23
LR_EN_C0.001_l10.05   temp_val_AUC=0.6851  val_AUC_drop=0.6633  val_Recall=0.5957  val_Precision=0.2822  val_F1=0.3830  threshold=0.16
LR_EN_C0.001_l10.1    temp_val_AUC=0.6843  val_AUC_drop=0.6615  val_Recall=0.5809  val_Precision=0.2848  val_F1=0.3822  threshold=0.16
LR_EN_C0.001_l10.2    temp_val_AUC=0.6833  val_AUC_drop=0.6584  val_Recall=0.5542  val_Precision=0.2904  val_F1=0.3811  threshold=0.16
LR_L2_C0.0001         temp_val_AUC=0.6823  val_AUC_drop=0.6608  val_Recall=0.6628  val_Precision=0.2654  val_F1=0.3791  threshold=0.39
LR_L2_C0.01           temp_val_AUC=0.6823  val_AUC_drop=0.6678  val_Recall=0.6441  val_Precision=0.2701  val_F1=0.3806  threshold=0.16
LR_L1_C0.1            temp_val_AUC=0.6820  val_AUC_drop=0.6614  val_Recall=0.6310  val_Precision=0.2735  val_F1=0.3816  threshold=0.15
LR_EN_C0.001_l10.3    temp_val_AU

In [69]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# L2
grid_search_lr_l2 = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=2000),
    {'C': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0], 'penalty': ['l2'], 'solver': ['liblinear']},
    cv=3, scoring='roc_auc', n_jobs=-1, verbose=1
)
grid_search_lr_l2.fit(X_train_t_sc, y_train_t)
print(f"L2 Best Score: {grid_search_lr_l2.best_score_:.4f}")
print(f"L2 Best Params: {grid_search_lr_l2.best_params_}")

# L1
grid_search_lr_l1 = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=2000),
    {'C': [0.0001, 0.001, 0.01, 1.0, 10.0, 100.0], 'penalty': ['l1'], 'solver': ['liblinear']},
    cv=3, scoring='roc_auc', n_jobs=-1, verbose=1
)
grid_search_lr_l1.fit(X_train_t_sc, y_train_t)
print(f"L1 Best Score: {grid_search_lr_l1.best_score_:.4f}")
print(f"L1 Best Params: {grid_search_lr_l1.best_params_}")

# ElasticNet
grid_search_lr_en = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=2000),
    {'C': [0.0001, 0.001, 1.01], 'penalty': ['elasticnet'], 'solver': ['saga'], 'l1_ratio': [0.05, 0.1, 0.2, 0.3, 0.5, 0.7]},
    cv=3, scoring='roc_auc', n_jobs=-1, verbose=1
)
grid_search_lr_en.fit(X_train_t_sc, y_train_t)
print(f"EN Best Score: {grid_search_lr_en.best_score_:.4f}")
print(f"EN Best Params: {grid_search_lr_en.best_params_}")

Fitting 3 folds for each of 7 candidates, totalling 21 fits
L2 Best Score: 0.6727
L2 Best Params: {'C': 1.0, 'penalty': 'l2', 'solver': 'liblinear'}
Fitting 3 folds for each of 6 candidates, totalling 18 fits
L1 Best Score: 0.6727
L1 Best Params: {'C': 1.0, 'penalty': 'l1', 'solver': 'liblinear'}
Fitting 3 folds for each of 18 candidates, totalling 54 fits
EN Best Score: 0.6727
EN Best Params: {'C': 1.01, 'l1_ratio': 0.7, 'penalty': 'elasticnet', 'solver': 'saga'}


In [70]:
import pandas as pd

for name, gs in [('L2', grid_search_lr_l2), ('L1', grid_search_lr_l1), ('EN', grid_search_lr_en)]:
    print(f"=== {name} Parameter Impact Analysis ===")
    results = pd.DataFrame(gs.cv_results_)
    for p in ['param_C', 'param_penalty', 'param_l1_ratio']:
        try:
            grouped = results.groupby(p)['mean_test_score'].mean()
            spread = grouped.max() - grouped.min()
            print(f"{str(p):25s} | score spread: {spread:.4f}")
            print(grouped.to_string())
            print()
        except:
            pass

=== L2 Parameter Impact Analysis ===
param_C                   | score spread: 0.0155
param_C
0.0001      0.657198
0.0010      0.665294
0.0100      0.670881
0.1000      0.672594
1.0000      0.672680
10.0000     0.672677
100.0000    0.672679

param_penalty             | score spread: 0.0000
param_penalty
l2    0.669144

=== L1 Parameter Impact Analysis ===
param_C                   | score spread: 0.1727
param_C
0.0001      0.500000
0.0010      0.500000
0.0100      0.666297
1.0000      0.672698
10.0000     0.672675
100.0000    0.672680

param_penalty             | score spread: 0.0000
param_penalty
l1    0.614058

=== EN Parameter Impact Analysis ===
param_C                   | score spread: 0.1462
param_C
0.0001    0.526513
0.0010    0.661528
1.0100    0.672677

param_penalty             | score spread: 0.0000
param_penalty
elasticnet    0.620239

param_l1_ratio            | score spread: 0.0547
param_l1_ratio
0.05    0.665417
0.10    0.612307
0.20    0.611542
0.30    0.610702
0.50    

## 3.2.2 LR Bayesian Search

In [71]:
# Optuna hyperparameter tuning for Logistic Regression, optimizing penalty type (L1/L2/ElasticNet), C, and l1_ratio over 40 trials.
def objective_lr(trial):
    penalty = trial.suggest_categorical('penalty', ['l2', 'l1', 'elasticnet'])
    C = trial.suggest_float('C', 1e-4, 1e-2, log=True)

    if penalty == 'elasticnet':
        l1_ratio = trial.suggest_float('l1_ratio', 0.05, 0.3)
        solver = 'saga'
    elif penalty == 'l1':
        l1_ratio = None
        solver = 'liblinear'
    else:
        l1_ratio = None
        solver = 'liblinear'

    kwargs = dict(C=C, penalty=penalty, solver=solver, class_weight='balanced', max_iter=2000, random_state=42)
    if l1_ratio is not None:
        kwargs['l1_ratio'] = l1_ratio

    model = LogisticRegression(**kwargs)
    model.fit(X_train_t_sc, y_train_t)

    yearly_aucs = {}
    for yr, (X_yr, y_yr, X_yr_sc, _) in yearly_tests.items():
        yp = model.predict_proba(X_yr_sc)[:, 1]
        yearly_aucs[yr] = roc_auc_score(y_yr, yp)

    auc_drop = compute_auc_score(yearly_aucs, VAL_YEARS)

    y_prob_val = model.predict_proba(X_val_t_sc)[:, 1]
    thresholds = np.arange(0.05, 0.50, 0.01)
    best_thr = max(thresholds, key=lambda t: f1_score(y_val_t, (y_prob_val >= t).astype(int), zero_division=0))
    val_f1 = f1_score(y_val_t, (y_prob_val >= best_thr).astype(int), zero_division=0)

    return w_auc * auc_drop + w_f1 * val_f1
study_lr = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_lr.optimize(objective_lr, n_trials=80, show_progress_bar=True)

[I 2026-05-09 10:04:40,341] A new study created in memory with name: no-name-035f703d-e3b9-43c6-9426-f815cc3db8d5


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-05-09 10:04:40,595] Trial 0 finished with value: 0.5155228058099783 and parameters: {'penalty': 'l1', 'C': 0.0015751320499779737}. Best is trial 0 with value: 0.5155228058099783.
[I 2026-05-09 10:04:40,874] Trial 1 finished with value: 0.5242583499835478 and parameters: {'penalty': 'l2', 'C': 0.005399484409787433}. Best is trial 1 with value: 0.5242583499835478.
[I 2026-05-09 10:04:41,143] Trial 2 finished with value: 0.5225456982310585 and parameters: {'penalty': 'l1', 'C': 0.008706020878304856}. Best is trial 1 with value: 0.5242583499835478.
[I 2026-05-09 10:04:41,419] Trial 3 finished with value: 0.5192757514420203 and parameters: {'penalty': 'l2', 'C': 0.00023270677083837802}. Best is trial 1 with value: 0.5242583499835478.
[I 2026-05-09 10:04:41,687] Trial 4 finished with value: 0.4962883157936615 and parameters: {'penalty': 'l1', 'C': 0.0003823475224675188}. Best is trial 1 with value: 0.5242583499835478.
[I 2026-05-09 10:04:41,961] Trial 5 finished with value: 0.5224392

In [72]:
# best
top_trials_lr = sorted(study_lr.trials, key=lambda t: t.value if t.value else -999, reverse=True)[:5]

LR_MODEL_CONFIGS = {}
for i, t in enumerate(top_trials_lr):
    name = f'LR_optuna_{i+1}'
    params = t.params.copy()
    penalty  = params.pop('penalty')
    C        = params.pop('C')
    l1_ratio = params.pop('l1_ratio', None)
    solver   = 'saga' if penalty == 'elasticnet' else 'liblinear'

    kwargs = dict(C=C, penalty=penalty, solver=solver, max_iter=2000, random_state=42)
    if l1_ratio is not None:
        kwargs['l1_ratio'] = l1_ratio

    LR_MODEL_CONFIGS[name] = (LogisticRegression(**kwargs), 'scaled')

# evaluation
lr_results = evaluate_model_configs(LR_MODEL_CONFIGS)

# val set
print_selection_table(lr_results, sort_key='val_auc', reverse=True, title='LR - Val Set', mode='val')
print_selection_table(lr_results, sort_key='val_auc_drop', reverse=False, title='LR - Val Robustness (TSS)', mode='val')
print_selection_table(lr_results, sort_key='val_f1', reverse=True, title='LR - Val F1', mode='val')

# best summary
print_best_summary(lr_results, prefix='LR_', title='Logistic Regression Selection Summary')

# Hyperparameter print
print_optuna_results(study_lr, top_trials_lr, title='LR Optuna Hyperparameter Results')


=== LR - Val Set ===
LR_optuna_1           temp_val_AUC=0.6843  val_AUC_drop=0.6648  val_Recall=0.5801  val_Precision=0.2860  val_F1=0.3831  threshold=0.16
LR_optuna_5           temp_val_AUC=0.6838  val_AUC_drop=0.6669  val_Recall=0.6433  val_Precision=0.2709  val_F1=0.3813  threshold=0.17
LR_optuna_4           temp_val_AUC=0.6838  val_AUC_drop=0.6669  val_Recall=0.6423  val_Precision=0.2711  val_F1=0.3813  threshold=0.17
LR_optuna_3           temp_val_AUC=0.6837  val_AUC_drop=0.6634  val_Recall=0.6272  val_Precision=0.2749  val_F1=0.3823  threshold=0.15
LR_optuna_2           temp_val_AUC=0.6837  val_AUC_drop=0.6641  val_Recall=0.6284  val_Precision=0.2747  val_F1=0.3823  threshold=0.15

=== LR - Val Robustness (TSS) ===
LR_optuna_3           temp_val_AUC=0.6837  val_AUC_drop=0.6634  val_Recall=0.6272  val_Precision=0.2749  val_F1=0.3823  threshold=0.15
LR_optuna_2           temp_val_AUC=0.6837  val_AUC_drop=0.6641  val_Recall=0.6284  val_Precision=0.2747  val_F1=0.3823  threshold=0.1

## 3.3 Decision Tree Test Code (Find Best)


## 3.2.1 DT Grid Search

In [73]:
# A dictionary defining DT model configurations by systematically combining max_depth, min_samples_leaf, min_samples_split,
# criterion (gini/entropy), and class_weight (balanced).
DT_MODEL_CONFIGS = {
    # 1) max_depth sweep
    'DTree_d3':   (DecisionTreeClassifier(max_depth=3,  random_state=42), 'scaled'),
    'DTree_d4':   (DecisionTreeClassifier(max_depth=4,  random_state=42), 'scaled'),
    'DTree_d5':   (DecisionTreeClassifier(max_depth=5,  random_state=42), 'scaled'),
    'DTree_d7':   (DecisionTreeClassifier(max_depth=7,  random_state=42), 'scaled'),
    'DTree_d10':  (DecisionTreeClassifier(max_depth=10, random_state=42), 'scaled'),

    'DTree_d3':           (DecisionTreeClassifier(max_depth=3,  random_state=42), 'scaled'),
    'DTree_d3_ent':       (DecisionTreeClassifier(max_depth=3,  criterion='entropy', random_state=42), 'scaled'),
    'DTree_d3_bal':       (DecisionTreeClassifier(max_depth=3,  class_weight='balanced', random_state=42), 'scaled'),

    'DTree_d4':           (DecisionTreeClassifier(max_depth=4,  random_state=42), 'scaled'),
    'DTree_d4_ent':       (DecisionTreeClassifier(max_depth=4,  criterion='entropy', random_state=42), 'scaled'),
    'DTree_d4_bal':       (DecisionTreeClassifier(max_depth=4,  class_weight='balanced', random_state=42), 'scaled'),

    'DTree_d5':           (DecisionTreeClassifier(max_depth=5,  random_state=42), 'scaled'),
    'DTree_d5_ent':       (DecisionTreeClassifier(max_depth=5,  criterion='entropy', random_state=42), 'scaled'),
    'DTree_d5_bal':       (DecisionTreeClassifier(max_depth=5,  class_weight='balanced', random_state=42), 'scaled'),

    'DTree_d7':           (DecisionTreeClassifier(max_depth=7,  random_state=42), 'scaled'),
    'DTree_d7_ent':       (DecisionTreeClassifier(max_depth=7,  criterion='entropy', random_state=42), 'scaled'),
    'DTree_d7_bal':       (DecisionTreeClassifier(max_depth=7,  class_weight='balanced', random_state=42), 'scaled'),

    'DTree_d10':          (DecisionTreeClassifier(max_depth=10, random_state=42), 'scaled'),
    'DTree_d10_ent':      (DecisionTreeClassifier(max_depth=10, criterion='entropy', random_state=42), 'scaled'),
    'DTree_d10_bal':      (DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=42), 'scaled'),

    # 2) min_samples_leaf sweep (d3, d5, d7 각각)
    'DTree_d3_leaf5':   (DecisionTreeClassifier(max_depth=3, min_samples_leaf=5,  random_state=42), 'scaled'),
    'DTree_d3_leaf20':  (DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=42), 'scaled'),
    'DTree_d3_leaf50':  (DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=42), 'scaled'),
    'DTree_d5_leaf5':   (DecisionTreeClassifier(max_depth=5, min_samples_leaf=5,  random_state=42), 'scaled'),
    'DTree_d5_leaf20':  (DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=42), 'scaled'),
    'DTree_d5_leaf50':  (DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=42), 'scaled'),
    'DTree_d7_leaf5':   (DecisionTreeClassifier(max_depth=7, min_samples_leaf=5,  random_state=42), 'scaled'),
    'DTree_d7_leaf20':  (DecisionTreeClassifier(max_depth=7, min_samples_leaf=20, random_state=42), 'scaled'),
    'DTree_d7_leaf50':  (DecisionTreeClassifier(max_depth=7, min_samples_leaf=50, random_state=42), 'scaled'),

    # min_samples_split sweep (d3, d5, d7 각각)
    'DTree_d3_split10':  (DecisionTreeClassifier(max_depth=3, min_samples_split=10, random_state=42), 'scaled'),
    'DTree_d3_split20':  (DecisionTreeClassifier(max_depth=3, min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d3_split50':  (DecisionTreeClassifier(max_depth=3, min_samples_split=50, random_state=42), 'scaled'),
    'DTree_d5_split10':  (DecisionTreeClassifier(max_depth=5, min_samples_split=10, random_state=42), 'scaled'),
    'DTree_d5_split20':  (DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d5_split50':  (DecisionTreeClassifier(max_depth=5, min_samples_split=50, random_state=42), 'scaled'),
    'DTree_d7_split10':  (DecisionTreeClassifier(max_depth=7, min_samples_split=10, random_state=42), 'scaled'),
    'DTree_d7_split20':  (DecisionTreeClassifier(max_depth=7, min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d7_split50':  (DecisionTreeClassifier(max_depth=7, min_samples_split=50, random_state=42), 'scaled'),

    # depth=3, leaf=5
    'DTree_d3_leaf5_split10':  (DecisionTreeClassifier(max_depth=3, min_samples_leaf=5,  min_samples_split=10, random_state=42), 'scaled'),
    'DTree_d3_leaf5_split20':  (DecisionTreeClassifier(max_depth=3, min_samples_leaf=5,  min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d3_leaf5_split50':  (DecisionTreeClassifier(max_depth=3, min_samples_leaf=5,  min_samples_split=50, random_state=42), 'scaled'),

    # depth=3, leaf=20
    'DTree_d3_leaf20_split10': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, min_samples_split=10, random_state=42), 'scaled'),
    'DTree_d3_leaf20_split20': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d3_leaf20_split50': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, min_samples_split=50, random_state=42), 'scaled'),

    # depth=3, leaf=50
    'DTree_d3_leaf50_split10': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, min_samples_split=10, random_state=42), 'scaled'),
    'DTree_d3_leaf50_split20': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d3_leaf50_split50': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, min_samples_split=50, random_state=42), 'scaled'),

    # depth=4, leaf=5
    'DTree_d4_leaf5_split10':  (DecisionTreeClassifier(max_depth=4, min_samples_leaf=5,  min_samples_split=10, random_state=42), 'scaled'),
    'DTree_d4_leaf5_split20':  (DecisionTreeClassifier(max_depth=4, min_samples_leaf=5,  min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d4_leaf5_split50':  (DecisionTreeClassifier(max_depth=4, min_samples_leaf=5,  min_samples_split=50, random_state=42), 'scaled'),

    # depth=4, leaf=20
    'DTree_d4_leaf20_split10': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, min_samples_split=10, random_state=42), 'scaled'),
    'DTree_d4_leaf20_split20': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d4_leaf20_split50': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, min_samples_split=50, random_state=42), 'scaled'),

    # depth=4, leaf=50
    'DTree_d4_leaf50_split10': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, min_samples_split=10, random_state=42), 'scaled'),
    'DTree_d4_leaf50_split20': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d4_leaf50_split50': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, min_samples_split=50, random_state=42), 'scaled'),

    # depth=5, leaf=5
    'DTree_d5_leaf5_split10':  (DecisionTreeClassifier(max_depth=5, min_samples_leaf=5,  min_samples_split=10, random_state=42), 'scaled'),
    'DTree_d5_leaf5_split20':  (DecisionTreeClassifier(max_depth=5, min_samples_leaf=5,  min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d5_leaf5_split50':  (DecisionTreeClassifier(max_depth=5, min_samples_leaf=5,  min_samples_split=50, random_state=42), 'scaled'),

    # depth=5, leaf=20
    'DTree_d5_leaf20_split10': (DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, min_samples_split=10, random_state=42), 'scaled'),
    'DTree_d5_leaf20_split20': (DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d5_leaf20_split50': (DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, min_samples_split=50, random_state=42), 'scaled'),

    # depth=5, leaf=50
    'DTree_d5_leaf50_split10': (DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, min_samples_split=10, random_state=42), 'scaled'),
    'DTree_d5_leaf50_split20': (DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d5_leaf50_split50': (DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, min_samples_split=50, random_state=42), 'scaled'),

    # depth=6, leaf=5
    'DTree_d6_leaf5_split10':  (DecisionTreeClassifier(max_depth=6, min_samples_leaf=5,  min_samples_split=10, random_state=42), 'scaled'),
    'DTree_d6_leaf5_split20':  (DecisionTreeClassifier(max_depth=6, min_samples_leaf=5,  min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d6_leaf5_split50':  (DecisionTreeClassifier(max_depth=6, min_samples_leaf=5,  min_samples_split=50, random_state=42), 'scaled'),

    # depth=6, leaf=20
    'DTree_d6_leaf20_split10': (DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, min_samples_split=10, random_state=42), 'scaled'),
    'DTree_d6_leaf20_split20': (DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d6_leaf20_split50': (DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, min_samples_split=50, random_state=42), 'scaled'),

    # depth=6, leaf=50
    'DTree_d6_leaf50_split10': (DecisionTreeClassifier(max_depth=6, min_samples_leaf=50, min_samples_split=10, random_state=42), 'scaled'),
    'DTree_d6_leaf50_split20': (DecisionTreeClassifier(max_depth=6, min_samples_leaf=50, min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d6_leaf50_split50': (DecisionTreeClassifier(max_depth=6, min_samples_leaf=50, min_samples_split=50, random_state=42), 'scaled'),

    # depth=3, entropy
    'DTree_d3_leaf5_split10_ent':  (DecisionTreeClassifier(max_depth=3, min_samples_leaf=5,  min_samples_split=10, criterion='entropy', random_state=42), 'scaled'),
    'DTree_d3_leaf5_split20_ent':  (DecisionTreeClassifier(max_depth=3, min_samples_leaf=5,  min_samples_split=20, criterion='entropy', random_state=42), 'scaled'),
    'DTree_d3_leaf5_split50_ent':  (DecisionTreeClassifier(max_depth=3, min_samples_leaf=5,  min_samples_split=50, criterion='entropy', random_state=42), 'scaled'),
    'DTree_d3_leaf20_split10_ent': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, min_samples_split=10, criterion='entropy', random_state=42), 'scaled'),
    'DTree_d3_leaf20_split20_ent': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, min_samples_split=20, criterion='entropy', random_state=42), 'scaled'),
    'DTree_d3_leaf20_split50_ent': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, min_samples_split=50, criterion='entropy', random_state=42), 'scaled'),
    'DTree_d3_leaf50_split10_ent': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, min_samples_split=10, criterion='entropy', random_state=42), 'scaled'),
    'DTree_d3_leaf50_split20_ent': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, min_samples_split=20, criterion='entropy', random_state=42), 'scaled'),
    'DTree_d3_leaf50_split50_ent': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, min_samples_split=50, criterion='entropy', random_state=42), 'scaled'),

    # depth=3, balanced
    'DTree_d3_leaf5_split10_bal':  (DecisionTreeClassifier(max_depth=3, min_samples_leaf=5,  min_samples_split=10, class_weight='balanced', random_state=42), 'scaled'),
    'DTree_d3_leaf5_split20_bal':  (DecisionTreeClassifier(max_depth=3, min_samples_leaf=5,  min_samples_split=20, class_weight='balanced', random_state=42), 'scaled'),
    'DTree_d3_leaf5_split50_bal':  (DecisionTreeClassifier(max_depth=3, min_samples_leaf=5,  min_samples_split=50, class_weight='balanced', random_state=42), 'scaled'),
    'DTree_d3_leaf20_split10_bal': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, min_samples_split=10, class_weight='balanced', random_state=42), 'scaled'),
    'DTree_d3_leaf20_split20_bal': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, min_samples_split=20, class_weight='balanced', random_state=42), 'scaled'),
    'DTree_d3_leaf20_split50_bal': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, min_samples_split=50, class_weight='balanced', random_state=42), 'scaled'),
    'DTree_d3_leaf50_split10_bal': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, min_samples_split=10, class_weight='balanced', random_state=42), 'scaled'),
    'DTree_d3_leaf50_split20_bal': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, min_samples_split=20, class_weight='balanced', random_state=42), 'scaled'),
    'DTree_d3_leaf50_split50_bal': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, min_samples_split=50, class_weight='balanced', random_state=42), 'scaled'),

    # depth=4, entropy
    'DTree_d4_leaf5_split10_ent':  (DecisionTreeClassifier(max_depth=4, min_samples_leaf=5,  min_samples_split=10, criterion='entropy', random_state=42), 'scaled'),
    'DTree_d4_leaf5_split20_ent':  (DecisionTreeClassifier(max_depth=4, min_samples_leaf=5,  min_samples_split=20, criterion='entropy', random_state=42), 'scaled'),
    'DTree_d4_leaf5_split50_ent':  (DecisionTreeClassifier(max_depth=4, min_samples_leaf=5,  min_samples_split=50, criterion='entropy', random_state=42), 'scaled'),
    'DTree_d4_leaf20_split10_ent': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, min_samples_split=10, criterion='entropy', random_state=42), 'scaled'),
    'DTree_d4_leaf20_split20_ent': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, min_samples_split=20, criterion='entropy', random_state=42), 'scaled'),
    'DTree_d4_leaf20_split50_ent': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, min_samples_split=50, criterion='entropy', random_state=42), 'scaled'),
    'DTree_d4_leaf50_split10_ent': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, min_samples_split=10, criterion='entropy', random_state=42), 'scaled'),
    'DTree_d4_leaf50_split20_ent': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, min_samples_split=20, criterion='entropy', random_state=42), 'scaled'),
    'DTree_d4_leaf50_split50_ent': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, min_samples_split=50, criterion='entropy', random_state=42), 'scaled'),

    # depth=4, balanced
    'DTree_d4_leaf5_split10_bal':  (DecisionTreeClassifier(max_depth=4, min_samples_leaf=5,  min_samples_split=10, class_weight='balanced', random_state=42), 'scaled'),
    'DTree_d4_leaf5_split20_bal':  (DecisionTreeClassifier(max_depth=4, min_samples_leaf=5,  min_samples_split=20, class_weight='balanced', random_state=42), 'scaled'),
    'DTree_d4_leaf5_split50_bal':  (DecisionTreeClassifier(max_depth=4, min_samples_leaf=5,  min_samples_split=50, class_weight='balanced', random_state=42), 'scaled'),
    'DTree_d4_leaf20_split10_bal': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, min_samples_split=10, class_weight='balanced', random_state=42), 'scaled'),
    'DTree_d4_leaf20_split20_bal': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, min_samples_split=20, class_weight='balanced', random_state=42), 'scaled'),
    'DTree_d4_leaf20_split50_bal': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, min_samples_split=50, class_weight='balanced', random_state=42), 'scaled'),
    'DTree_d4_leaf50_split10_bal': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, min_samples_split=10, class_weight='balanced', random_state=42), 'scaled'),
    'DTree_d4_leaf50_split20_bal': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, min_samples_split=20, class_weight='balanced', random_state=42), 'scaled'),
    'DTree_d4_leaf50_split50_bal': (DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, min_samples_split=50, class_weight='balanced', random_state=42), 'scaled'),

    # combo (leaf + split + d3, d5, d7 각각)
    'DTree_d3_leaf5_split20':  (DecisionTreeClassifier(max_depth=3, min_samples_leaf=5,  min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d3_leaf20_split50': (DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, min_samples_split=50, random_state=42), 'scaled'),
    'DTree_d5_leaf5_split20':  (DecisionTreeClassifier(max_depth=5, min_samples_leaf=5,  min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d5_leaf20_split50': (DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, min_samples_split=50, random_state=42), 'scaled'),
    'DTree_d7_leaf5_split20':  (DecisionTreeClassifier(max_depth=7, min_samples_leaf=5,  min_samples_split=20, random_state=42), 'scaled'),
    'DTree_d7_leaf20_split50': (DecisionTreeClassifier(max_depth=7, min_samples_leaf=20, min_samples_split=50, random_state=42), 'scaled'),
}

print(f'Total DT model configs: {len(DT_MODEL_CONFIGS)}')

Total DT model configs: 107


In [74]:
dt_results = evaluate_model_configs(DT_MODEL_CONFIGS)

# (val set)
print_selection_table(dt_results, sort_key='val_auc', reverse=True, title='DT - Val Set', mode='val')
print_selection_table(dt_results, sort_key='val_auc_drop', reverse=False, title='DT - Val Robustness', mode='val')

# (test set)
print_selection_table(dt_results, sort_key='test_auc', reverse=True, title='DT - Test Set (Final)', mode='test')
print_selection_table(dt_results, sort_key='test_auc_drop', reverse=False, title='DT - Test Robustness (Final)', mode='test')

# (best summary)
print_best_summary(dt_results, prefix='DTree_', title='Decision Tree Selection Summary')


=== DT - Val Set ===
DTree_d4              temp_val_AUC=0.6719  val_AUC_drop=0.6529  val_Recall=0.6146  val_Precision=0.2664  val_F1=0.3717  threshold=0.13
DTree_d4_leaf5_split10  temp_val_AUC=0.6719  val_AUC_drop=0.6529  val_Recall=0.6146  val_Precision=0.2664  val_F1=0.3717  threshold=0.13
DTree_d4_leaf5_split20  temp_val_AUC=0.6719  val_AUC_drop=0.6529  val_Recall=0.6146  val_Precision=0.2664  val_F1=0.3717  threshold=0.13
DTree_d4_leaf5_split50  temp_val_AUC=0.6719  val_AUC_drop=0.6529  val_Recall=0.6146  val_Precision=0.2664  val_F1=0.3717  threshold=0.13
DTree_d4_leaf20_split10  temp_val_AUC=0.6719  val_AUC_drop=0.6529  val_Recall=0.6146  val_Precision=0.2664  val_F1=0.3717  threshold=0.13
DTree_d4_leaf20_split20  temp_val_AUC=0.6719  val_AUC_drop=0.6529  val_Recall=0.6146  val_Precision=0.2664  val_F1=0.3717  threshold=0.13
DTree_d4_leaf20_split50  temp_val_AUC=0.6719  val_AUC_drop=0.6529  val_Recall=0.6146  val_Precision=0.2664  val_F1=0.3717  threshold=0.13
DTree_d4_leaf50_sp

In [75]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
import time

param_grid_dt = {
    'max_depth': [3, 4, 5, 7, 10],
    'min_samples_leaf': [1, 5, 20, 50],
    'min_samples_split': [2, 10, 20, 50],
    'criterion': ['gini', 'entropy'],
    'class_weight': [None, 'balanced'],
}

total_combinations = 1
for v in param_grid_dt.values():
    total_combinations *= len(v)
print(f"total: {total_combinations:,} x cv=3 = {total_combinations*3:,} fits")

grid_search_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_dt,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=3
)

start = time.time()
grid_search_dt.fit(X_train_t_sc, y_train_t)
elapsed = time.time() - start

print(f"\nTotal time: {elapsed/60:.1f} min")
print(f"Best Score: {grid_search_dt.best_score_:.4f}")
print(f"Best Params: {grid_search_dt.best_params_}")

total: 320 x cv=3 = 960 fits
Fitting 3 folds for each of 320 candidates, totalling 960 fits

Total time: 0.1 min
Best Score: 0.6594
Best Params: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 4, 'min_samples_leaf': 1, 'min_samples_split': 50}


In [76]:
import pandas as pd

results_dt = pd.DataFrame(grid_search_dt.cv_results_)

params = ['param_max_depth', 'param_min_samples_leaf', 'param_min_samples_split',
          'param_criterion', 'param_class_weight']

print("=== Parameter Impact Analysis ===")
for p in params:
    try:
        grouped = results_dt.groupby(p)['mean_test_score'].mean()
        spread = grouped.max() - grouped.min()
        print(f"{str(p):30s} | score spread: {spread:.4f}")
        print(grouped.to_string())
        print()
    except:
        print(f"{str(p):30s} | not found, skipped")
        print()

=== Parameter Impact Analysis ===
param_max_depth                | score spread: 0.0462
param_max_depth
3     0.654867
4     0.656910
5     0.655169
7     0.636823
10    0.610691

param_min_samples_leaf         | score spread: 0.0021
param_min_samples_leaf
1     0.643216
5     0.642352
20    0.641941
50    0.644059

param_min_samples_split        | score spread: 0.0008
param_min_samples_split
2     0.642662
10    0.642595
20    0.642884
50    0.643427

param_criterion                | score spread: 0.0011
param_criterion
entropy    0.643431
gini       0.642353

param_class_weight             | score spread: 0.0000
param_class_weight
balanced    0.64269



## 3.2.2 DT Bayesian Search

In [77]:
# Optuna hyperparameter tuning for Decision Tree, optimizing TRP over 60 trials by searching
# max_depth, min_samples_leaf, min_samples_split, and criterion.
def objective_dt(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 5),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 50),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
    }

    model = DecisionTreeClassifier(**params, random_state=42)
    model.fit(X_train_t_sc, y_train_t)

    yearly_aucs = {}
    for yr, (X_yr, y_yr, X_yr_sc, _) in yearly_tests.items():
        yp = model.predict_proba(X_yr_sc)[:, 1]
        yearly_aucs[yr] = roc_auc_score(y_yr, yp)
    auc_drop = compute_auc_score(yearly_aucs, VAL_YEARS)

    y_prob_val = model.predict_proba(X_val_t_sc)[:, 1]
    thresholds = np.arange(0.05, 0.50, 0.01)
    best_thr = max(thresholds, key=lambda t: f1_score(y_val_t, (y_prob_val >= t).astype(int), zero_division=0))
    val_f1 = f1_score(y_val_t, (y_prob_val >= best_thr).astype(int), zero_division=0)

    return w_auc * auc_drop + w_f1 * val_f1
study_dt = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_dt.optimize(objective_dt, n_trials=80, show_progress_bar=True)

[I 2026-05-09 10:07:40,286] A new study created in memory with name: no-name-e9c24db1-c4a3-4ced-b60a-41edea7c2a95


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-05-09 10:07:40,554] Trial 0 finished with value: 0.512300802128161 and parameters: {'max_depth': 4, 'min_samples_leaf': 48, 'min_samples_split': 37, 'criterion': 'gini'}. Best is trial 0 with value: 0.512300802128161.
[I 2026-05-09 10:07:40,802] Trial 1 finished with value: 0.5109928965630844 and parameters: {'max_depth': 3, 'min_samples_leaf': 3, 'min_samples_split': 44, 'criterion': 'entropy'}. Best is trial 0 with value: 0.512300802128161.
[I 2026-05-09 10:07:41,045] Trial 2 finished with value: 0.5071588165794062 and parameters: {'max_depth': 3, 'min_samples_leaf': 49, 'min_samples_split': 42, 'criterion': 'gini'}. Best is trial 0 with value: 0.512300802128161.
[I 2026-05-09 10:07:41,287] Trial 3 finished with value: 0.5071588165794062 and parameters: {'max_depth': 3, 'min_samples_leaf': 16, 'min_samples_split': 27, 'criterion': 'gini'}. Best is trial 0 with value: 0.512300802128161.
[I 2026-05-09 10:07:41,546] Trial 4 finished with value: 0.5114425375276195 and parameters:

In [78]:
# best
top_trials_dt = sorted(study_dt.trials, key=lambda t: t.value if t.value else -999, reverse=True)[:5]

DT_MODEL_CONFIGS = {}
for i, t in enumerate(top_trials_dt):
    name = f'DTree_optuna_{i+1}'
    DT_MODEL_CONFIGS[name] = (
        DecisionTreeClassifier(**t.params, random_state=42), 'raw'
    )

# evaluation
dt_results = evaluate_model_configs(DT_MODEL_CONFIGS)

# val set
print_selection_table(dt_results, sort_key='val_auc', reverse=True, title='DT - Val Set', mode='val')
print_selection_table(dt_results, sort_key='val_auc_drop', reverse=False, title='DT - Val Robustness (TSS)', mode='val')
print_selection_table(dt_results, sort_key='val_f1', reverse=True, title='DT - Val F1', mode='val')

# best summary
print_best_summary(dt_results, prefix='DTree_', title='Decision Tree Selection Summary')

# Hyperparameter print
print_optuna_results(study_dt, top_trials_dt, title='DT Optuna Hyperparameter Results')


=== DT - Val Set ===
DTree_optuna_1        temp_val_AUC=0.6689  val_AUC_drop=0.6535  val_Recall=0.6709  val_Precision=0.2602  val_F1=0.3749  threshold=0.15
DTree_optuna_2        temp_val_AUC=0.6689  val_AUC_drop=0.6535  val_Recall=0.6709  val_Precision=0.2602  val_F1=0.3749  threshold=0.15
DTree_optuna_3        temp_val_AUC=0.6689  val_AUC_drop=0.6535  val_Recall=0.6709  val_Precision=0.2602  val_F1=0.3749  threshold=0.15
DTree_optuna_4        temp_val_AUC=0.6689  val_AUC_drop=0.6535  val_Recall=0.6709  val_Precision=0.2602  val_F1=0.3749  threshold=0.15
DTree_optuna_5        temp_val_AUC=0.6689  val_AUC_drop=0.6535  val_Recall=0.6709  val_Precision=0.2602  val_F1=0.3749  threshold=0.15

=== DT - Val Robustness (TSS) ===
DTree_optuna_1        temp_val_AUC=0.6689  val_AUC_drop=0.6535  val_Recall=0.6709  val_Precision=0.2602  val_F1=0.3749  threshold=0.15
DTree_optuna_2        temp_val_AUC=0.6689  val_AUC_drop=0.6535  val_Recall=0.6709  val_Precision=0.2602  val_F1=0.3749  threshold=0.1

## 3.4 Random Forest Test Code (Find Best)


## 3.4.1 RF Grid Search

In [79]:
# A dictionary defining RF model configurations by systematically combining n_estimators, max_depth, min_samples_leaf, min_samples_split, max_features, and class_weight.
RF_MODEL_CONFIGS = {
    # n_estimators sweep
    'RF_n50_d3':    (RandomForestClassifier(n_estimators=50,  max_depth=3, random_state=42, n_jobs=-1), 'scaled'),
    'RF_n100_d3':   (RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d3':   (RandomForestClassifier(n_estimators=200, max_depth=3, random_state=42, n_jobs=-1), 'scaled'),
    'RF_n300_d3':   (RandomForestClassifier(n_estimators=300, max_depth=3, random_state=42, n_jobs=-1), 'scaled'),
    'RF_n50_d5':    (RandomForestClassifier(n_estimators=50,  max_depth=5, random_state=42, n_jobs=-1), 'scaled'),
    'RF_n100_d5':   (RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d5':   (RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42, n_jobs=-1), 'scaled'),
    'RF_n300_d5':   (RandomForestClassifier(n_estimators=300, max_depth=5, random_state=42, n_jobs=-1), 'scaled'),

    # max_depth sweep
    'RF_n200_d4':   (RandomForestClassifier(n_estimators=200, max_depth=4,  random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d6':   (RandomForestClassifier(n_estimators=200, max_depth=6,  random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d8':   (RandomForestClassifier(n_estimators=200, max_depth=8,  random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d12':  (RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1), 'scaled'),

    # min_samples_leaf sweep
    'RF_n200_d3_leaf5':   (RandomForestClassifier(n_estimators=200, max_depth=3, min_samples_leaf=5,  random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d3_leaf10':  (RandomForestClassifier(n_estimators=200, max_depth=3, min_samples_leaf=10, random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d3_leaf20':  (RandomForestClassifier(n_estimators=200, max_depth=3, min_samples_leaf=20, random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d5_leaf5':   (RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=5,  random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d5_leaf10':  (RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=10, random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d5_leaf20':  (RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=20, random_state=42, n_jobs=-1), 'scaled'),

    # max_features sweep
    'RF_n200_d3_sqrt':  (RandomForestClassifier(n_estimators=200, max_depth=3, max_features='sqrt', random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d3_log2':  (RandomForestClassifier(n_estimators=200, max_depth=3, max_features='log2', random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d3_half':  (RandomForestClassifier(n_estimators=200, max_depth=3, max_features=0.5,    random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d5_sqrt':  (RandomForestClassifier(n_estimators=200, max_depth=5, max_features='sqrt', random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d5_log2':  (RandomForestClassifier(n_estimators=200, max_depth=5, max_features='log2', random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d5_half':  (RandomForestClassifier(n_estimators=200, max_depth=5, max_features=0.5,    random_state=42, n_jobs=-1), 'scaled'),

    # min_samples_split sweep
    'RF_n200_d3_split5':   (RandomForestClassifier(n_estimators=200, max_depth=3, min_samples_split=5,  random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d3_split10':  (RandomForestClassifier(n_estimators=200, max_depth=3, min_samples_split=10, random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d3_split20':  (RandomForestClassifier(n_estimators=200, max_depth=3, min_samples_split=20, random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d5_split5':   (RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_split=5,  random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d5_split10':  (RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_split=10, random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d5_split20':  (RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_split=20, random_state=42, n_jobs=-1), 'scaled'),

    # class_weight sweep
    'RF_n100_d3_balanced': (RandomForestClassifier(n_estimators=200, max_depth=3, class_weight='balanced', random_state=42, n_jobs=-1), 'scaled'),
    'RF_n100_d5_balanced': (RandomForestClassifier(n_estimators=200, max_depth=3, class_weight='balanced_subsample', random_state=42, n_jobs=-1), 'scaled'),

    # combo (n_estimators + leaf + features, d3, d5)
    'RF_n200_d3_leaf5_sqrt':  (RandomForestClassifier(n_estimators=200, max_depth=3, min_samples_leaf=5,  max_features='sqrt', random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d3_leaf10_sqrt': (RandomForestClassifier(n_estimators=200, max_depth=3, min_samples_leaf=10, max_features='sqrt', random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d5_leaf5_sqrt':  (RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=5,  max_features='sqrt', random_state=42, n_jobs=-1), 'scaled'),
    'RF_n200_d5_leaf10_sqrt': (RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=10, max_features='sqrt', random_state=42, n_jobs=-1), 'scaled'),
}

print(f'Total RF model configs: {len(RF_MODEL_CONFIGS)}')

Total RF model configs: 36


In [80]:
rf_results = evaluate_model_configs(RF_MODEL_CONFIGS)

# (val set)
print_selection_table(rf_results, sort_key='val_auc', reverse=True, title='RF - Val Set', mode='val')
print_selection_table(rf_results, sort_key='val_auc_drop', reverse=False, title='RF - Val Robustness', mode='val')

# (test set)
print_selection_table(rf_results, sort_key='test_auc', reverse=True, title='RF - Test Set (Final)', mode='test')
print_selection_table(rf_results, sort_key='test_auc_drop', reverse=False, title='RF - Test Robustness (Final)', mode='test')

# (best summary)
print_best_summary(rf_results, prefix='RF_', title='Random Forest Selection Summary')


=== RF - Val Set ===
RF_n200_d3_half       temp_val_AUC=0.6819  val_AUC_drop=0.6644  val_Recall=0.5703  val_Precision=0.2840  val_F1=0.3792  threshold=0.15
RF_n200_d5_leaf20     temp_val_AUC=0.6812  val_AUC_drop=0.6651  val_Recall=0.6161  val_Precision=0.2746  val_F1=0.3799  threshold=0.15
RF_n200_d5_leaf10     temp_val_AUC=0.6812  val_AUC_drop=0.6650  val_Recall=0.6227  val_Precision=0.2735  val_F1=0.3801  threshold=0.15
RF_n200_d5_leaf10_sqrt  temp_val_AUC=0.6812  val_AUC_drop=0.6650  val_Recall=0.6227  val_Precision=0.2735  val_F1=0.3801  threshold=0.15
RF_n200_d5_leaf5      temp_val_AUC=0.6811  val_AUC_drop=0.6652  val_Recall=0.6264  val_Precision=0.2719  val_F1=0.3792  threshold=0.15
RF_n200_d5_leaf5_sqrt  temp_val_AUC=0.6811  val_AUC_drop=0.6652  val_Recall=0.6264  val_Precision=0.2719  val_F1=0.3792  threshold=0.15
RF_n200_d6            temp_val_AUC=0.6809  val_AUC_drop=0.6654  val_Recall=0.5722  val_Precision=0.2838  val_F1=0.3794  threshold=0.16
RF_n200_d5_half       temp_val

In [85]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
import time

param_dist_rf = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [3, 4, 5, 6, 8, 12],
    'min_samples_leaf': [1, 5, 10, 20],
    'min_samples_split': [2, 5, 10, 20],
    'max_features': ['sqrt', 'log2', 0.5],
    'class_weight': [None, 'balanced', 'balanced_subsample'],
}

total_combinations = 1
for v in param_dist_rf.values():
    total_combinations *= len(v)
print(f"total: {total_combinations:,} x cv=3 = {total_combinations*3:,} fits")
print(f"sampling: 1000 x cv=3 = {1000*3:,} fits")

random_search_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_dist_rf,
    n_iter=1000,
    cv=3,
    scoring='roc_auc',
    n_jobs=22,
    verbose=1,
    random_state=42
)

start = time.time()
random_search_rf.fit(X_train_t_sc, y_train_t)
elapsed = time.time() - start

print(f"\nTotal time: {elapsed/60:.1f} min")
print(f"Best Score: {random_search_rf.best_score_:.4f}")
print(f"Best Params: {random_search_rf.best_params_}")

total: 3,456 x cv=3 = 10,368 fits
sampling: 1000 x cv=3 = 3,000 fits
Fitting 3 folds for each of 1000 candidates, totalling 3000 fits


KeyboardInterrupt: 

In [ ]:
import pandas as pd

results_rf = pd.DataFrame(random_search_rf.cv_results_)

params = ['param_n_estimators', 'param_max_depth', 'param_min_samples_leaf',
          'param_min_samples_split', 'param_max_features', 'param_class_weight']

print("=== RF Parameter Impact Analysis ===")
for p in params:
    try:
        grouped = results_rf.groupby(p)['mean_test_score'].mean()
        spread = grouped.max() - grouped.min()
        print(f"{str(p):30s} | score spread: {spread:.4f}")
        print(grouped.to_string())
        print()
    except:
        print(f"{str(p):30s} | not found, skipped")
        print()

## 3.4.2 RF Bayesian Search

In [86]:
# Optuna hyperparameter tuning for Random Forest,
# optimizing TRP over 50 trials by searching n_estimators, max_depth, min_samples_leaf, and max_features.
def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 5),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 0.5]),
    }

    model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
    model.fit(X_train_t_sc, y_train_t)

    yearly_aucs = {}
    for yr, (X_yr, y_yr, X_yr_sc, _) in yearly_tests.items():
        yp = model.predict_proba(X_yr_sc)[:, 1]
        yearly_aucs[yr] = roc_auc_score(y_yr, yp)
    auc_drop = compute_auc_score(yearly_aucs, VAL_YEARS)

    y_prob_val = model.predict_proba(X_val_t_sc)[:, 1]
    thresholds = np.arange(0.05, 0.50, 0.01)
    best_thr = max(thresholds, key=lambda t: f1_score(y_val_t, (y_prob_val >= t).astype(int), zero_division=0))
    val_f1 = f1_score(y_val_t, (y_prob_val >= best_thr).astype(int), zero_division=0)

    return w_auc * auc_drop + w_f1 * val_f1
study_rf = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_rf.optimize(objective_rf, n_trials=80, show_progress_bar=True)

[I 2026-05-09 10:32:38,397] A new study created in memory with name: no-name-30ec722f-4536-4be3-9343-3c8ae4d7a516


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-05-09 10:32:39,252] Trial 0 finished with value: 0.5224925658258136 and parameters: {'n_estimators': 106, 'max_depth': 5, 'min_samples_leaf': 15, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5224925658258136.
[I 2026-05-09 10:32:40,032] Trial 1 finished with value: 0.520416771860017 and parameters: {'n_estimators': 73, 'max_depth': 3, 'min_samples_leaf': 18, 'max_features': 0.5}. Best is trial 0 with value: 0.5224925658258136.
[I 2026-05-09 10:32:40,641] Trial 2 finished with value: 0.5228479174013322 and parameters: {'n_estimators': 53, 'max_depth': 5, 'min_samples_leaf': 17, 'max_features': 'sqrt'}. Best is trial 2 with value: 0.5228479174013322.
[I 2026-05-09 10:32:41,330] Trial 3 finished with value: 0.5181927911894249 and parameters: {'n_estimators': 77, 'max_depth': 3, 'min_samples_leaf': 11, 'max_features': 'sqrt'}. Best is trial 2 with value: 0.5228479174013322.
[I 2026-05-09 10:32:42,339] Trial 4 finished with value: 0.52101212758488 and parameters: {'n_estim

In [87]:
# best
top_trials_rf = sorted(study_rf.trials, key=lambda t: t.value if t.value else -999, reverse=True)[:5]

RF_MODEL_CONFIGS = {}
for i, t in enumerate(top_trials_rf):
    name = f'RF_optuna_{i+1}'
    RF_MODEL_CONFIGS[name] = (
        RandomForestClassifier(**t.params, random_state=42, n_jobs=-1), 'raw'
    )

# evaluation
rf_results = evaluate_model_configs(RF_MODEL_CONFIGS)

# val set
print_selection_table(rf_results, sort_key='val_auc', reverse=True, title='RF - Val Set', mode='val')
print_selection_table(rf_results, sort_key='val_auc_drop', reverse=False, title='RF - Val Robustness (TSS)', mode='val')
print_selection_table(rf_results, sort_key='val_f1', reverse=True, title='RF - Val F1', mode='val')

# best summary
print_best_summary(rf_results, prefix='RF_', title='Random Forest Selection Summary')

# Hyperparameter print
print_optuna_results(study_rf, top_trials_rf, title='RF Optuna Hyperparameter Results')


=== RF - Val Set ===
RF_optuna_3           temp_val_AUC=0.6823  val_AUC_drop=0.6670  val_Recall=0.6190  val_Precision=0.2756  val_F1=0.3814  threshold=0.15
RF_optuna_1           temp_val_AUC=0.6823  val_AUC_drop=0.6672  val_Recall=0.6179  val_Precision=0.2759  val_F1=0.3815  threshold=0.15
RF_optuna_5           temp_val_AUC=0.6822  val_AUC_drop=0.6670  val_Recall=0.6196  val_Precision=0.2753  val_F1=0.3812  threshold=0.15
RF_optuna_2           temp_val_AUC=0.6822  val_AUC_drop=0.6671  val_Recall=0.6157  val_Precision=0.2762  val_F1=0.3813  threshold=0.15
RF_optuna_4           temp_val_AUC=0.6819  val_AUC_drop=0.6670  val_Recall=0.6134  val_Precision=0.2767  val_F1=0.3814  threshold=0.15

=== RF - Val Robustness (TSS) ===
RF_optuna_4           temp_val_AUC=0.6819  val_AUC_drop=0.6670  val_Recall=0.6134  val_Precision=0.2767  val_F1=0.3814  threshold=0.15
RF_optuna_5           temp_val_AUC=0.6822  val_AUC_drop=0.6670  val_Recall=0.6196  val_Precision=0.2753  val_F1=0.3812  threshold=0.1

## 3.5.1 NN Grid Search

In [88]:
# A dictionary defining MLP model configurations by sweeping hidden_layer_sizes,
# alpha (L2 regularization), and learning_rate_init across various combinations.
from sklearn.neural_network import MLPClassifier

NN_MODEL_CONFIGS = {
    # hidden_layer_sizes sweep
    'NN_64':                  (MLPClassifier(hidden_layer_sizes=(64,),       max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128':                 (MLPClassifier(hidden_layer_sizes=(128,),      max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_64_32':               (MLPClassifier(hidden_layer_sizes=(64, 32),    max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_64':              (MLPClassifier(hidden_layer_sizes=(128, 64),   max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_64_32':           (MLPClassifier(hidden_layer_sizes=(128,64,32), max_iter=200, early_stopping=True, random_state=42), 'scaled'),

    # alpha sweep (64)
    'NN_64_a0001':            (MLPClassifier(hidden_layer_sizes=(64,),       alpha=0.0001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_64_a001':             (MLPClassifier(hidden_layer_sizes=(64,),       alpha=0.001,  max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_64_a01':              (MLPClassifier(hidden_layer_sizes=(64,),       alpha=0.01,   max_iter=200, early_stopping=True, random_state=42), 'scaled'),

    # alpha sweep (128)
    'NN_128_a0001':           (MLPClassifier(hidden_layer_sizes=(128,),      alpha=0.0001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_a001':            (MLPClassifier(hidden_layer_sizes=(128,),      alpha=0.001,  max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_a01':             (MLPClassifier(hidden_layer_sizes=(128,),      alpha=0.01,   max_iter=200, early_stopping=True, random_state=42), 'scaled'),

    # alpha sweep (64_32)
    'NN_64_32_a0001':         (MLPClassifier(hidden_layer_sizes=(64, 32),    alpha=0.0001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_64_32_a001':          (MLPClassifier(hidden_layer_sizes=(64, 32),    alpha=0.001,  max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_64_32_a01':           (MLPClassifier(hidden_layer_sizes=(64, 32),    alpha=0.01,   max_iter=200, early_stopping=True, random_state=42), 'scaled'),

    # alpha sweep (128_64)
    'NN_128_64_a0001':        (MLPClassifier(hidden_layer_sizes=(128, 64),   alpha=0.0001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_64_a001':         (MLPClassifier(hidden_layer_sizes=(128, 64),   alpha=0.001,  max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_64_a01':          (MLPClassifier(hidden_layer_sizes=(128, 64),   alpha=0.01,   max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_64':        (MLPClassifier(hidden_layer_sizes=(128, 64),   alpha=0.0001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_64':         (MLPClassifier(hidden_layer_sizes=(128, 64),   alpha=0.001,  max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_64':          (MLPClassifier(hidden_layer_sizes=(128, 64),   alpha=0.01,   max_iter=200, early_stopping=True, random_state=42), 'scaled'),

    # alpha sweep (128_64_32)
    'NN_128_64_32_a0001':     (MLPClassifier(hidden_layer_sizes=(128,64,32), alpha=0.0001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_64_32_a001':      (MLPClassifier(hidden_layer_sizes=(128,64,32), alpha=0.001,  max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_64_32_a01':       (MLPClassifier(hidden_layer_sizes=(128,64,32), alpha=0.01,   max_iter=200, early_stopping=True, random_state=42), 'scaled'),

    # # learning_rate_init sweep (64)
    'NN_64_lr001':            (MLPClassifier(hidden_layer_sizes=(64,),       learning_rate_init=0.001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_64_lr01':             (MLPClassifier(hidden_layer_sizes=(64,),       learning_rate_init=0.01,  max_iter=200, early_stopping=True, random_state=42), 'scaled'),

    # learning_rate_init sweep (128)
    'NN_128_lr001':           (MLPClassifier(hidden_layer_sizes=(128,),      learning_rate_init=0.001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_lr01':            (MLPClassifier(hidden_layer_sizes=(128,),      learning_rate_init=0.01,  max_iter=200, early_stopping=True, random_state=42), 'scaled'),

    # learning_rate_init sweep (128_64)
    'NN_128_64_lr001':        (MLPClassifier(hidden_layer_sizes=(128, 64),   learning_rate_init=0.001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_64_lr01':         (MLPClassifier(hidden_layer_sizes=(128, 64),   learning_rate_init=0.01,  max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_64_32_lr001':        (MLPClassifier(hidden_layer_sizes=(128, 64),   learning_rate_init=0.001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_64_32_lr01':         (MLPClassifier(hidden_layer_sizes=(128, 64),   learning_rate_init=0.01,  max_iter=200, early_stopping=True, random_state=42), 'scaled'),

    # combo
    'NN_128_64_a001_lr001':   (MLPClassifier(hidden_layer_sizes=(128, 64),   alpha=0.001, learning_rate_init=0.001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_64_a01_lr001':    (MLPClassifier(hidden_layer_sizes=(128, 64),   alpha=0.01,  learning_rate_init=0.001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_64_a001_lr01':    (MLPClassifier(hidden_layer_sizes=(128, 64),   alpha=0.001, learning_rate_init=0.01,  max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_64_32_a001_lr001':(MLPClassifier(hidden_layer_sizes=(128,64,32), alpha=0.001, learning_rate_init=0.001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),




    # alpha (L2 regularization) sweep
    'NN_64_a0001':    (MLPClassifier(hidden_layer_sizes=(64,),  alpha=0.0001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_64_a001':     (MLPClassifier(hidden_layer_sizes=(64,),  alpha=0.001,  max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_64_a01':      (MLPClassifier(hidden_layer_sizes=(64,),  alpha=0.01,   max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_a0001':   (MLPClassifier(hidden_layer_sizes=(128,), alpha=0.0001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_a001':    (MLPClassifier(hidden_layer_sizes=(128,), alpha=0.001,  max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_a01':     (MLPClassifier(hidden_layer_sizes=(128,), alpha=0.01,   max_iter=200, early_stopping=True, random_state=42), 'scaled'),

    # learning_rate_init sweep
    'NN_64_lr001':    (MLPClassifier(hidden_layer_sizes=(64,),  learning_rate_init=0.001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_64_lr01':     (MLPClassifier(hidden_layer_sizes=(64,),  learning_rate_init=0.01,  max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_lr001':   (MLPClassifier(hidden_layer_sizes=(128,), learning_rate_init=0.001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_lr01':    (MLPClassifier(hidden_layer_sizes=(128,), learning_rate_init=0.01,  max_iter=200, early_stopping=True, random_state=42), 'scaled'),

    # combo
    'NN_128_64_a001_lr001': (MLPClassifier(hidden_layer_sizes=(128,64), alpha=0.001, learning_rate_init=0.001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_64_a01_lr001':  (MLPClassifier(hidden_layer_sizes=(128,64), alpha=0.01,  learning_rate_init=0.001, max_iter=200, early_stopping=True, random_state=42), 'scaled'),
    'NN_128_64_a001_lr01':  (MLPClassifier(hidden_layer_sizes=(128,64), alpha=0.001, learning_rate_init=0.01,  max_iter=200, early_stopping=True, random_state=42), 'scaled'),

    }

print(f'Total NN model configs: {len(NN_MODEL_CONFIGS)}')

Total NN model configs: 32


In [89]:
nn_results = evaluate_model_configs(NN_MODEL_CONFIGS)
print_selection_table(nn_results, sort_key='val_auc', reverse=True, title='NN - Val Set', mode='val')
print_selection_table(nn_results, sort_key='val_auc_drop', reverse=False, title='NN - Val Robustness (TSS)', mode='val')
print_selection_table(nn_results, sort_key='val_f1', reverse=True, title='NN - Val F1', mode='val')
print_best_summary(nn_results, prefix='NN_', title='Neural Network Selection Summary')


=== NN - Val Set ===
NN_64_lr01            temp_val_AUC=0.6757  val_AUC_drop=0.6673  val_Recall=0.6293  val_Precision=0.2704  val_F1=0.3783  threshold=0.17
NN_128_64_32_a01      temp_val_AUC=0.6736  val_AUC_drop=0.6651  val_Recall=0.6382  val_Precision=0.2642  val_F1=0.3737  threshold=0.17
NN_128_64_lr01        temp_val_AUC=0.6733  val_AUC_drop=0.6627  val_Recall=0.6507  val_Precision=0.2631  val_F1=0.3747  threshold=0.11
NN_128_64_32_lr01     temp_val_AUC=0.6733  val_AUC_drop=0.6627  val_Recall=0.6507  val_Precision=0.2631  val_F1=0.3747  threshold=0.11
NN_128_64_32_a001     temp_val_AUC=0.6733  val_AUC_drop=0.6652  val_Recall=0.6054  val_Precision=0.2720  val_F1=0.3753  threshold=0.18
NN_128_64_32_a001_lr001  temp_val_AUC=0.6733  val_AUC_drop=0.6652  val_Recall=0.6054  val_Precision=0.2720  val_F1=0.3753  threshold=0.18
NN_128_64_a001_lr01   temp_val_AUC=0.6733  val_AUC_drop=0.6617  val_Recall=0.5824  val_Precision=0.2760  val_F1=0.3745  threshold=0.19
NN_128_64_32          temp_val

In [90]:
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier
import time

param_grid_nn = {
    'hidden_layer_sizes': [(64,), (128,), (64, 32), (128, 64), (128, 64, 32)],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.0001, 0.001, 0.01],
    'max_iter': [200],
    'early_stopping': [True],
}

total_combinations = 1
for v in param_grid_nn.values():
    total_combinations *= len(v)
print(f"total: {total_combinations:,} x cv=3 = {total_combinations*3:,} fits")

grid_search_nn = GridSearchCV(
    MLPClassifier(random_state=42),
    param_grid_nn,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=3
)

start = time.time()
grid_search_nn.fit(X_train_t_sc, y_train_t)
elapsed = time.time() - start

print(f"\nTotal time: {elapsed/60:.1f} min")
print(f"Best Score: {grid_search_nn.best_score_:.4f}")
print(f"Best Params: {grid_search_nn.best_params_}")

total: 45 x cv=3 = 135 fits
Fitting 3 folds for each of 45 candidates, totalling 135 fits

Total time: 0.1 min
Best Score: 0.6691
Best Params: {'alpha': 0.001, 'early_stopping': True, 'hidden_layer_sizes': (128,), 'learning_rate_init': 0.01, 'max_iter': 200}


In [91]:
import pandas as pd

results_nn = pd.DataFrame(grid_search_nn.cv_results_)

params = ['param_hidden_layer_sizes', 'param_alpha', 'param_learning_rate_init']

print("=== NN Parameter Impact Analysis ===")
for p in params:
    try:
        grouped = results_nn.groupby(p)['mean_test_score'].mean()
        spread = grouped.max() - grouped.min()
        print(f"{str(p):30s} | score spread: {spread:.4f}")
        print(grouped.to_string())
        print()
    except:
        print(f"{str(p):30s} | not found, skipped")
        print()

=== NN Parameter Impact Analysis ===
param_hidden_layer_sizes       | score spread: 0.0419
param_hidden_layer_sizes
(64,)            0.619332
(64, 32)         0.620639
(128,)           0.627872
(128, 64)        0.602742
(128, 64, 32)    0.644646

param_alpha                    | score spread: 0.0021
param_alpha
0.0001    0.621786
0.0010    0.623919
0.0100    0.623435

param_learning_rate_init       | score spread: 0.0893
param_learning_rate_init
0.0001    0.569332
0.0010    0.641156
0.0100    0.658652



## 3.5.2 NN Bayesian Search

In [92]:
# Optuna hyperparameter tuning for MLP, optimizing TRP over 60 trials
# by searching hidden_layer_sizes, alpha, and learning_rate_init.
def objective_nn(trial):
    params = {
        'hidden_layer_sizes': trial.suggest_categorical('hidden_layer_sizes',
            [(128,), (128, 64), (128, 64, 32)]),
        'alpha': trial.suggest_float('alpha', 1e-4, 1e-1, log=True),
        'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
        'max_iter': 200,
        'early_stopping': True,
        'random_state': 42,
    }

    model = MLPClassifier(**params)
    model.fit(X_train_t_sc, y_train_t)

    yearly_aucs = {}
    for yr, (X_yr, y_yr, X_yr_sc, _) in yearly_tests.items():
        yp = model.predict_proba(X_yr_sc)[:, 1]
        yearly_aucs[yr] = roc_auc_score(y_yr, yp)
    auc_drop = compute_auc_score(yearly_aucs, VAL_YEARS)

    y_prob_val = model.predict_proba(X_val_t_sc)[:, 1]
    thresholds = np.arange(0.05, 0.50, 0.01)
    best_thr = max(thresholds, key=lambda t: f1_score(y_val_t, (y_prob_val >= t).astype(int), zero_division=0))
    val_f1 = f1_score(y_val_t, (y_prob_val >= best_thr).astype(int), zero_division=0)

    return w_auc * auc_drop + w_f1 * val_f1

study_nn = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_nn.optimize(objective_nn, n_trials=60, show_progress_bar=True)

[I 2026-05-09 10:36:08,129] A new study created in memory with name: no-name-56208d5f-1c5d-4521-959f-634a080f19db


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-05-09 10:36:09,629] Trial 0 finished with value: 0.42342693068203885 and parameters: {'hidden_layer_sizes': (128, 64), 'alpha': 0.006251373574521752, 'learning_rate_init': 0.0002051338263087451}. Best is trial 0 with value: 0.42342693068203885.
[I 2026-05-09 10:36:12,695] Trial 1 finished with value: 0.5217041224086315 and parameters: {'hidden_layer_sizes': (128, 64, 32), 'alpha': 0.006358358856676255, 'learning_rate_init': 0.0026070247583707684}. Best is trial 1 with value: 0.5217041224086315.
[I 2026-05-09 10:36:14,231] Trial 2 finished with value: 0.43166429464232825 and parameters: {'hidden_layer_sizes': (128, 64), 'alpha': 0.0004335281794951569, 'learning_rate_init': 0.0002310201887845295}. Best is trial 1 with value: 0.5217041224086315.
[I 2026-05-09 10:36:16,693] Trial 3 finished with value: 0.5255573285119816 and parameters: {'hidden_layer_sizes': (128, 64, 32), 'alpha': 0.0019762189340280074, 'learning_rate_init': 0.0003823475224675188}. Best is trial 3 with value: 0.5

In [93]:
# best
top_trials_nn = sorted(study_nn.trials, key=lambda t: t.value if t.value else -999, reverse=True)[:5]

NN_MODEL_CONFIGS = {}
for i, t in enumerate(top_trials_nn):
    name = f'NN_optuna_{i+1}'
    NN_MODEL_CONFIGS[name] = (MLPClassifier(**t.params, max_iter=200, early_stopping=True, random_state=42), 'scaled')

# evaluation
nn_results = evaluate_model_configs(NN_MODEL_CONFIGS)

# val set
print_selection_table(nn_results, sort_key='val_auc', reverse=True, title='NN - Val Set', mode='val')
print_selection_table(nn_results, sort_key='val_auc_drop', reverse=False, title='NN - Val Robustness (TSS)', mode='val')
print_selection_table(nn_results, sort_key='val_f1', reverse=True, title='NN - Val F1', mode='val')

# best summary
print_best_summary(nn_results, prefix='NN_', title='Neural Network Selection Summary')

# Hyperparameter print
print_optuna_results(study_nn, top_trials_nn, title='NN Optuna Hyperparameter Results')


=== NN - Val Set ===
NN_optuna_1           temp_val_AUC=0.6816  val_AUC_drop=0.6699  val_Recall=0.6151  val_Precision=0.2762  val_F1=0.3812  threshold=0.16
NN_optuna_3           temp_val_AUC=0.6806  val_AUC_drop=0.6657  val_Recall=0.6214  val_Precision=0.2754  val_F1=0.3817  threshold=0.15
NN_optuna_2           temp_val_AUC=0.6803  val_AUC_drop=0.6657  val_Recall=0.5748  val_Precision=0.2857  val_F1=0.3817  threshold=0.16
NN_optuna_4           temp_val_AUC=0.6798  val_AUC_drop=0.6661  val_Recall=0.6063  val_Precision=0.2774  val_F1=0.3807  threshold=0.15
NN_optuna_5           temp_val_AUC=0.6788  val_AUC_drop=0.6650  val_Recall=0.5759  val_Precision=0.2848  val_F1=0.3811  threshold=0.16

=== NN - Val Robustness (TSS) ===
NN_optuna_5           temp_val_AUC=0.6788  val_AUC_drop=0.6650  val_Recall=0.5759  val_Precision=0.2848  val_F1=0.3811  threshold=0.16
NN_optuna_3           temp_val_AUC=0.6806  val_AUC_drop=0.6657  val_Recall=0.6214  val_Precision=0.2754  val_F1=0.3817  threshold=0.1

## 3.6.1 LGBM Grid Search

In [94]:
# A dictionary defining LightGBM model configurations by systematically combining n_estimators, max_depth, learning_rate, and num_leaves.
from lightgbm import LGBMClassifier

LGBM_MODEL_CONFIGS = {
    # n_estimators sweep
    # 'LGBM_n50_d3':   (LGBMClassifier(n_estimators=50,  max_depth=3, random_state=42, verbose=-1), 'scaled'),
    'LGBM_n100_d3':  (LGBMClassifier(n_estimators=100, max_depth=3, random_state=42, verbose=-1), 'scaled'),
    'LGBM_n200_d3':  (LGBMClassifier(n_estimators=200, max_depth=3, random_state=42, verbose=-1), 'scaled'),
    # 'LGBM_n50_d5':   (LGBMClassifier(n_estimators=50,  max_depth=5, random_state=42, verbose=-1), 'scaled'),
    'LGBM_n100_d5':  (LGBMClassifier(n_estimators=100, max_depth=5, random_state=42, verbose=-1), 'scaled'),
    'LGBM_n200_d5':  (LGBMClassifier(n_estimators=200, max_depth=5, random_state=42, verbose=-1), 'scaled'),

    # learning_rate sweep
    'LGBM_n100_d3_lr001':  (LGBMClassifier(n_estimators=100, max_depth=3, learning_rate=0.01,  random_state=42, verbose=-1), 'scaled'),
    'LGBM_n100_d3_lr005':  (LGBMClassifier(n_estimators=100, max_depth=3, learning_rate=0.05,  random_state=42, verbose=-1), 'scaled'),
    'LGBM_n100_d3_lr01':   (LGBMClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,   random_state=42, verbose=-1), 'scaled'),
    'LGBM_n100_d5_lr001':  (LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.01,  random_state=42, verbose=-1), 'scaled'),
    'LGBM_n100_d5_lr005':  (LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.05,  random_state=42, verbose=-1), 'scaled'),
    'LGBM_n100_d5_lr01':   (LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,   random_state=42, verbose=-1), 'scaled'),
    'LGBM_n200_d3_lr001':  (LGBMClassifier(n_estimators=100, max_depth=3, learning_rate=0.01,  random_state=42, verbose=-1), 'scaled'),
    'LGBM_n200_d3_lr005':  (LGBMClassifier(n_estimators=100, max_depth=3, learning_rate=0.05,  random_state=42, verbose=-1), 'scaled'),
    'LGBM_n200_d3_lr01':   (LGBMClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,   random_state=42, verbose=-1), 'scaled'),
    'LGBM_n200_d5_lr001':  (LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.01,  random_state=42, verbose=-1), 'scaled'),
    'LGBM_n200_d5_lr005':  (LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.05,  random_state=42, verbose=-1), 'scaled'),
    'LGBM_n200_d5_lr01':   (LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,   random_state=42, verbose=-1), 'scaled'),

    # num_leaves sweep
    'LGBM_n100_d3_l15':  (LGBMClassifier(n_estimators=100, max_depth=3, num_leaves=15, random_state=42, verbose=-1), 'scaled'),
    'LGBM_n100_d3_l31':  (LGBMClassifier(n_estimators=100, max_depth=3, num_leaves=31, random_state=42, verbose=-1), 'scaled'),
    'LGBM_n100_d3_l63':  (LGBMClassifier(n_estimators=100, max_depth=3, num_leaves=63, random_state=42, verbose=-1), 'scaled'),
    'LGBM_n100_d5_l15':  (LGBMClassifier(n_estimators=100, max_depth=5, num_leaves=15, random_state=42, verbose=-1), 'scaled'),
    'LGBM_n100_d5_l31':  (LGBMClassifier(n_estimators=100, max_depth=5, num_leaves=31, random_state=42, verbose=-1), 'scaled'),
    'LGBM_n100_d5_l63':  (LGBMClassifier(n_estimators=100, max_depth=5, num_leaves=63, random_state=42, verbose=-1), 'scaled'),

    # combo
    'LGBM_n200_d3_lr005_l31': (LGBMClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1), 'scaled'),
    'LGBM_n200_d5_lr005_l31': (LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1), 'scaled'),
    'LGBM_n200_d3_lr01_l15':  (LGBMClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,  num_leaves=15, random_state=42, verbose=-1), 'scaled'),
    'LGBM_n200_d5_lr01_l15':  (LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,  num_leaves=15, random_state=42, verbose=-1), 'scaled'),
}

print(f'Total LGBM model configs: {len(LGBM_MODEL_CONFIGS)}')

Total LGBM model configs: 26


In [95]:
lgbm_results = evaluate_model_configs(LGBM_MODEL_CONFIGS)
print_selection_table(lgbm_results, sort_key='val_auc', reverse=True, title='LGBM - Val Set', mode='val')
print_selection_table(lgbm_results, sort_key='val_auc_drop', reverse=False, title='LGBM - Val Robustness (TSS)', mode='val')
print_selection_table(lgbm_results, sort_key='val_f1', reverse=True, title='LGBM - Val F1', mode='val')
print_best_summary(lgbm_results, prefix='LGBM_', title='LightGBM Selection Summary')


=== LGBM - Val Set ===
LGBM_n100_d3_lr005    temp_val_AUC=0.6815  val_AUC_drop=0.6679  val_Recall=0.6005  val_Precision=0.2778  val_F1=0.3799  threshold=0.16
LGBM_n200_d3_lr005    temp_val_AUC=0.6815  val_AUC_drop=0.6679  val_Recall=0.6005  val_Precision=0.2778  val_F1=0.3799  threshold=0.16
LGBM_n200_d3_lr005_l31  temp_val_AUC=0.6811  val_AUC_drop=0.6683  val_Recall=0.5974  val_Precision=0.2778  val_F1=0.3793  threshold=0.16
LGBM_n100_d3          temp_val_AUC=0.6803  val_AUC_drop=0.6677  val_Recall=0.5966  val_Precision=0.2774  val_F1=0.3787  threshold=0.16
LGBM_n100_d3_lr01     temp_val_AUC=0.6803  val_AUC_drop=0.6677  val_Recall=0.5966  val_Precision=0.2774  val_F1=0.3787  threshold=0.16
LGBM_n200_d3_lr01     temp_val_AUC=0.6803  val_AUC_drop=0.6677  val_Recall=0.5966  val_Precision=0.2774  val_F1=0.3787  threshold=0.16
LGBM_n100_d3_l15      temp_val_AUC=0.6803  val_AUC_drop=0.6677  val_Recall=0.5966  val_Precision=0.2774  val_F1=0.3787  threshold=0.16
LGBM_n100_d3_l31      temp_va

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from lightgbm import LGBMClassifier
import time

param_dist_lgbm = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [15, 31, 63],
    'subsample': [0.6, 0.7, 0.9],
    'reg_alpha': [0.0, 1.0],
    'reg_lambda': [0.0, 1.0, 3.0],
}

total_combinations = 1
for v in param_dist_lgbm.values():
    total_combinations *= len(v)

N_ITER = 100

print(f"total: {total_combinations:,} x cv=3 = {total_combinations*3:,} fits")
print(f"sampling: {N_ITER} x cv=3 = {N_ITER*3:,} fits")

random_search_lgbm = RandomizedSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    param_dist_lgbm,
    n_iter=N_ITER,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

start = time.time()
random_search_lgbm.fit(X_train_t_sc, y_train_t)
elapsed = time.time() - start

print(f"\nTotal time: {elapsed/60:.1f} min")
print(f"Best Score: {random_search_lgbm.best_score_:.4f}")
print(f"Best Params: {random_search_lgbm.best_params_}")

total: 972 x cv=3 = 2,916 fits
sampling: 100 x cv=3 = 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits


In [ ]:
import pandas as pd

results_lgbm = pd.DataFrame(random_search_lgbm.cv_results_)

params = ['param_n_estimators', 'param_max_depth', 'param_learning_rate',
          'param_num_leaves', 'param_subsample', 'param_reg_alpha', 'param_reg_lambda']

print("=== LGBM Parameter Impact Analysis ===")
for p in params:
    try:
        grouped = results_lgbm.groupby(p)['mean_test_score'].mean()
        spread = grouped.max() - grouped.min()
        print(f"{str(p):30s} | score spread: {spread:.4f}")
        print(grouped.to_string())
        print()
    except:
        print(f"{str(p):30s} | not found, skipped")
        print()

## 3.6.2 LGBM Bayesian Search

In [ ]:
# Optuna hyperparameter tuning for LightGBM, optimizing TRP over 50 trials by searching
# n_estimators, learning_rate, max_depth, num_leaves, subsample, and regularization terms.
from lightgbm import LGBMClassifier

def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 5),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 3.0),
    }

    model = LGBMClassifier(**params, random_state=42, verbose=-1)
    model.fit(X_train_t_sc, y_train_t)

    yearly_aucs = {}
    for yr, (X_yr, y_yr, X_yr_sc, _) in yearly_tests.items():
        yp = model.predict_proba(X_yr_sc)[:, 1]
        yearly_aucs[yr] = roc_auc_score(y_yr, yp)
    auc_drop = compute_auc_score(yearly_aucs, VAL_YEARS)

    y_prob_val = model.predict_proba(X_val_t_sc)[:, 1]
    thresholds = np.arange(0.05, 0.50, 0.01)
    best_thr = max(thresholds, key=lambda t: f1_score(y_val_t, (y_prob_val >= t).astype(int), zero_division=0))
    val_f1 = f1_score(y_val_t, (y_prob_val >= best_thr).astype(int), zero_division=0)

    return w_auc * auc_drop + w_f1 * val_f1

study_lgbm = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_lgbm.optimize(objective_lgbm, n_trials=70, show_progress_bar=True)

In [ ]:
# best
top_trials_lgbm = sorted(study_lgbm.trials, key=lambda t: t.value if t.value else -999, reverse=True)[:5]

LGBM_MODEL_CONFIGS = {}
for i, t in enumerate(top_trials_lgbm):
    name = f'LGBM_optuna_{i+1}'
    LGBM_MODEL_CONFIGS[name] = (
        LGBMClassifier(**t.params, random_state=42, verbose=-1), 'raw'
    )

# evaluation
lgbm_results = evaluate_model_configs(LGBM_MODEL_CONFIGS)

# val set
print_selection_table(lgbm_results, sort_key='val_auc', reverse=True, title='LGBM - Val Set', mode='val')
print_selection_table(lgbm_results, sort_key='val_auc_drop', reverse=False, title='LGBM - Val Robustness (TSS)', mode='val')
print_selection_table(lgbm_results, sort_key='val_f1', reverse=True, title='LGBM - Val F1', mode='val')

# best summary
print_best_summary(lgbm_results, prefix='LGBM_', title='LightGBM Selection Summary')

# Hyperparameter print
print_optuna_results(study_lgbm, top_trials_lgbm, title='LGBM Optuna Hyperparameter Results')

## 4. Training & Temporal / Random AUC calculation


In [ ]:
all_results = {}
all_results.update(xgb_results)
all_results.update(lr_results)
all_results.update(dt_results)
all_results.update(rf_results)
all_results.update(nn_results)
all_results.update(lgbm_results)
print(f'Total combined model configs: {len(all_results)}')

In [ ]:
# (val set)
print_selection_table(all_results, sort_key='val_auc', reverse=True, title='All Models - Val Set', mode='val')
print_selection_table(all_results, sort_key='val_auc_drop', reverse=False, title='All Models - Val Robustness (TSS)', mode='val')
print_selection_table(all_results, sort_key='val_f1', reverse=True, title='All Models - Val F1', mode='val')

# (best summary)
print_best_summary(all_results, title='Best Model')

## 6. Fairness Calculation Function

In [ ]:
FAIRNESS_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]
EXPERIMENT_RESULTS = all_results
FAIRNESS_MODELS = list(EXPERIMENT_RESULTS.keys())
fairness_results = {}
val_meta = val_t.copy()

for name in FAIRNESS_MODELS:
    y_prob_t = EXPERIMENT_RESULTS[name]['temporal']['val_y_prob']
    best_thr = EXPERIMENT_RESULTS[name]['temporal']['threshold']

    # best_thr sweep range
    thr_low = max(0.05, best_thr - 0.10)
    thr_high = min(0.95, best_thr + 0.10)
    fairness_thresholds = np.round(np.linspace(thr_low, thr_high, 5), 2).tolist()

    fairness_results[name] = {
        'credit': compute_group_metrics_at_threshold(
            y_val_t, y_prob_t, ['short', 'mid', 'long'], val_meta['credit_group'], threshold=best_thr),
        'income': compute_group_metrics_at_threshold(
            y_val_t, y_prob_t, ['low', 'mid', 'high'], val_meta['income_group'], threshold=best_thr),
        'credit_sweep': compute_threshold_sweep(
            y_val_t, y_prob_t, ['short', 'mid', 'long'], val_meta['credit_group'], fairness_thresholds),
        'income_sweep': compute_threshold_sweep(
            y_val_t, y_prob_t, ['low', 'mid', 'high'], val_meta['income_group'], fairness_thresholds),
    }
print('Fairness computed for:', list(fairness_results.keys()))
print('Thresholds:', FAIRNESS_THRESHOLDS)

In [ ]:
rows = []
for name in FAIRNESS_MODELS:
    credit_df = fairness_results[name]['credit']
    income_df = fairness_results[name]['income']
    credit_sweep = fairness_results[name]['credit_sweep']
    income_sweep = fairness_results[name]['income_sweep']
    rows.append({
        'Model':                 name,
        'Temporal Val AUC':      round(EXPERIMENT_RESULTS[name]['temporal']['val_auc'], 4),
        'Val Accuracy':          round(EXPERIMENT_RESULTS[name]['temporal']['val_acc'], 4),
        'Val AUC Drop':          round(EXPERIMENT_RESULTS[name]['val_auc_drop'], 4),
        'Val Recall':            round(EXPERIMENT_RESULTS[name]['temporal']['val_recall'], 4),
        'Val Precision':         round(EXPERIMENT_RESULTS[name]['temporal']['val_precision'], 4),
        'Val F1':                round(EXPERIMENT_RESULTS[name]['temporal']['val_f1'], 4),
        'Credit FPR_disp@0.5':   round(credit_df['FPR'].max() - credit_df['FPR'].min(), 4),
        'Credit TPR_disp@0.5':   round(credit_df['TPR'].max() - credit_df['TPR'].min(), 4),
        'Credit PPR_disp@0.5':   round(credit_df['PPR'].max() - credit_df['PPR'].min(), 4),
        'Income FPR_disp@0.5':   round(income_df['FPR'].max() - income_df['FPR'].min(), 4),
        'Income TPR_disp@0.5':   round(income_df['TPR'].max() - income_df['TPR'].min(), 4),
        'Income PPR_disp@0.5':   round(income_df['PPR'].max() - income_df['PPR'].min(), 4),
        'Worst Credit FPR_disp': round(credit_sweep['FPR_disp'].max(), 4),
        'Worst Credit TPR_disp': round(credit_sweep['TPR_disp'].max(), 4),
        'Worst Credit PPR_disp': round(credit_sweep['PPR_disp'].max(), 4),
        'Worst Income FPR_disp': round(income_sweep['FPR_disp'].max(), 4),
        'Worst Income TPR_disp': round(income_sweep['TPR_disp'].max(), 4),
        'Worst Income PPR_disp': round(income_sweep['PPR_disp'].max(), 4),
    })

df_fairness_summary = pd.DataFrame(rows).sort_values('Temporal Val AUC', ascending=False)
# print('\n=== Threshold-Aware Fairness Summary ===')
# print(df_fairness_summary.to_string(index=False))

In [ ]:
df_score_raw = df_fairness_summary.copy()

df_score_raw['_auc_drop_norm'] = df_score_raw['Val AUC Drop']
df_score_raw['_auc_perf_norm'] = df_score_raw['Temporal Val AUC']
df_score_raw['_f1_norm'] = df_score_raw['Val F1']
df_score_raw['_credit_fpr_norm'] = df_score_raw['Worst Credit FPR_disp']
df_score_raw['_income_fpr_norm'] = df_score_raw['Worst Income FPR_disp']
df_score_raw['_credit_tpr_norm'] = df_score_raw['Worst Credit TPR_disp']
df_score_raw['_income_tpr_norm'] = df_score_raw['Worst Income TPR_disp']
df_score_raw['_credit_ppr_norm'] = df_score_raw['Worst Credit PPR_disp']
df_score_raw['_income_ppr_norm'] = df_score_raw['Worst Income PPR_disp']

df_score_raw['_fair_norm'] = (
    df_score_raw['_credit_fpr_norm'] + df_score_raw['_income_fpr_norm'] +
    df_score_raw['_credit_tpr_norm'] + df_score_raw['_income_tpr_norm']
) / 4

df_ranked_raw = df_score_raw[[
    'Model', 'Temporal Val AUC', 'Val AUC Drop', 'Val F1', '_fair_norm',
    'Worst Credit FPR_disp', 'Worst Income FPR_disp',
    'Worst Credit TPR_disp', 'Worst Income TPR_disp',
    'Worst Credit PPR_disp', 'Worst Income PPR_disp',
]].sort_values('Val AUC Drop', ascending=False)

print('=== Best Model Ranking (lower score = better) ===')
print(df_ranked_raw.to_string(index=False))
print(f'\n★ Best Model: {df_ranked_raw.iloc[0]["Model"]}')

In [ ]:
df_score_raw = df_fairness_summary.copy()

def minmax(col):
    return (col - col.min()) / (col.max() - col.min() + 1e-9)

df_score_raw['_auc_drop_norm'] = minmax(df_score_raw['Val AUC Drop'])
df_score_raw['_auc_perf_norm'] = minmax(df_score_raw['Temporal Val AUC'])
df_score_raw['_f1_norm'] = minmax(df_score_raw['Val F1'])

df_score_raw['_credit_fpr_norm'] = minmax(df_score_raw['Worst Credit FPR_disp'])
df_score_raw['_income_fpr_norm'] = minmax(df_score_raw['Worst Income FPR_disp'])
df_score_raw['_credit_tpr_norm'] = minmax(df_score_raw['Worst Credit TPR_disp'])
df_score_raw['_income_tpr_norm'] = minmax(df_score_raw['Worst Income TPR_disp'])
df_score_raw['_credit_ppr_norm'] = minmax(df_score_raw['Worst Credit PPR_disp'])
df_score_raw['_income_ppr_norm'] = minmax(df_score_raw['Worst Income PPR_disp'])

# df_score_raw['_auc_drop_norm'] = df_score_raw['Val AUC Drop']
# df_score_raw['_auc_perf_norm'] = df_score_raw['Temporal Val AUC']
# df_score_raw['_f1_norm'] = df_score_raw['Val F1']
# df_score_raw['_credit_fpr_norm'] = df_score_raw['Worst Credit FPR_disp']
# df_score_raw['_income_fpr_norm'] = df_score_raw['Worst Income FPR_disp']
# df_score_raw['_credit_tpr_norm'] = df_score_raw['Worst Credit TPR_disp']
# df_score_raw['_income_tpr_norm'] = df_score_raw['Worst Income TPR_disp']
# df_score_raw['_credit_ppr_norm'] = df_score_raw['Worst Credit PPR_disp']
# df_score_raw['_income_ppr_norm'] = df_score_raw['Worst Income PPR_disp']

df_score_raw['_fair_norm'] = (
    df_score_raw['_credit_fpr_norm'] + df_score_raw['_income_fpr_norm'] +
    df_score_raw['_credit_tpr_norm'] + df_score_raw['_income_tpr_norm']
) / 4

# Temporally Robust Performance (TRP)
df_score_raw['TRP'] = (df_score_raw['_auc_drop_norm']*0.5 + df_score_raw['_f1_norm']*0.5).round(4)

df_ranked_raw = df_score_raw[[
    'Model','TRP','Temporal Val AUC', 'Val AUC Drop', 'Val F1', '_fair_norm',
    'Worst Credit FPR_disp', 'Worst Income FPR_disp',
    'Worst Credit TPR_disp', 'Worst Income TPR_disp',
    'Worst Credit PPR_disp', 'Worst Income PPR_disp',
]].sort_values('TRP', ascending=False)

print('=== Best Model Ranking ===')
print(df_ranked_raw.to_string(index=False))
print(f'\n★ Best Model: {df_ranked_raw.iloc[0]["Model"]}')

## 5. Visualization